In [1]:
"""SESSION 28 v3 — CondInst R50-FPN with group-safe five-fold, multi-seed CV.

"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import multiprocessing
import statistics
import subprocess
import sys
import time
import types
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

# BEGIN EMBEDDED SESSION 28 HELPERS
# Generated snapshots; do not edit between these markers.
_SESSION_24_condinst_r101_dcn_cv_module = types.ModuleType('SESSION_24_condinst_r101_dcn_cv')
_SESSION_24_condinst_r101_dcn_cv_module.__file__ = '<embedded-SESSION_24_condinst_r101_dcn_cv>'
sys.modules['SESSION_24_condinst_r101_dcn_cv'] = _SESSION_24_condinst_r101_dcn_cv_module
exec(compile('"""SESSION 24 — genuine CondInst R101-DCNv2 with five-fold OOF CV.\n\nKaggle requirements\n-------------------\n* Accelerator: GPU (two T4 GPUs are used when available).\n* Internet: enabled on the first run for pinned packages and official weights.\n* Dataset: the Session-00 rebuilt dataset, preferably the group-disjoint V2\n  export made by ``kaggle_notebooks/KAGGLE_DATASET_REBUILD_V2.py``.\n\nThis implements Session 24 from ``model_strategy_analysis.md``: CondInst\'s\nper-instance dynamic mask convolutions, upgraded from ResNet-50 to a\nResNet-101 backbone with DCNv2 in C3-C5, and strict five-fold out-of-fold\nevaluation. It never substitutes another detector when CondInst/DCNv2 fails.\n"""\n\nfrom __future__ import annotations\n\n# %% 1. Imports and shared, already-audited CV utilities\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport multiprocessing\nimport os\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\n# Session 24 is a fully self-contained script.\n# No cross-session imports — SESSION_23 is not importable in a separate Kaggle cell.\n\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nos.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")\n\nSESSION_NAME = "session24_condinst_r101_dcn_cv"\nEXPECTED_CLASSES = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\nMMCV_INDEX = "https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html"\nTORCH_INDEX = "https://download.pytorch.org/whl/cu121"\nCONDINST_R50_URL = (\n    "https://download.openmmlab.com/mmdetection/v3.0/condinst/"\n    "condinst_r50_fpn_ms-poly-90k_coco_instance/"\n    "condinst_r50_fpn_ms-poly-90k_coco_instance_20221129_125223-4c186406.pth"\n)\n\n\ndef _pip(*args: str) -> None:\n    print(f"  [install] {\' \'.join(args)}")\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])\n\n\ndef _mmdet_tool(name: str) -> str:\n    import mmdet\n    root = Path(mmdet.__file__).resolve().parent\n    for candidate in [\n        root / ".mim" / "tools" / f"{name}.py",\n        root / "tools" / f"{name}.py",\n        root.parent / "tools" / f"{name}.py",\n    ]:\n        if candidate.is_file():\n            return str(candidate)\n    raise FileNotFoundError(f"MMDetection tools/{name}.py not found under {root}")\n\n\ndef _parse_folds(value: str) -> list[int]:\n    folds = sorted({int(part.strip()) for part in value.split(",") if part.strip()})\n    if not folds or any(fold not in range(5) for fold in folds):\n        raise ValueError("--folds must contain comma-separated values from 0 through 4")\n    return folds\n\n\ndef _run_training(config: Path, work_dir: Path, gpu_ids: list[str], resume: bool) -> None:\n    env = os.environ.copy()\n    env["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_ids)\n    train_tool = _mmdet_tool("train")\n    if len(gpu_ids) > 1:\n        cmd = [\n            sys.executable, "-m", "torch.distributed.run",\n            "--nproc_per_node", str(len(gpu_ids)),\n            "--master_port", str(random.randint(20000, 29999)),\n            train_tool, str(config), "--work-dir", str(work_dir),\n            "--launcher", "pytorch",\n        ]\n    else:\n        cmd = [sys.executable, train_tool, str(config), "--work-dir", str(work_dir)]\n    if resume:\n        cmd.append("--resume")\n    print(f"  [train] GPUs={gpu_ids}; {\' \'.join(cmd)}")\n    subprocess.check_call(cmd, env=env)\n\n\ndef _run_prediction(config: Path, checkpoint: Path, work_dir: Path, gpu_id: str) -> None:\n    env = os.environ.copy()\n    env["CUDA_VISIBLE_DEVICES"] = gpu_id\n    subprocess.check_call(\n        [sys.executable, _mmdet_tool("test"), str(config), str(checkpoint),\n         "--work-dir", str(work_dir)],\n        env=env,\n    )\n\n\ndef _best_checkpoint(work_dir: Path) -> Path | None:\n    # MMEngine writes best_coco_segm_mAP_50_iter_*.pth when save_best=\'coco/segm_mAP_50\'\n    candidates = sorted(\n        work_dir.glob("best_coco_segm_mAP_50_iter_*.pth"),\n        key=lambda p: p.stat().st_mtime,\n    )\n    if candidates:\n        return candidates[-1]\n    fallback = sorted(work_dir.glob("iter_*.pth"), key=lambda p: p.stat().st_mtime)\n    return fallback[-1] if fallback else None\n\n\ndef _polygon_area(xs: list[float], ys: list[float]) -> float:\n    return 0.5 * abs(sum(\n        xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i]\n        for i in range(len(xs))\n    ))\n\n\ndef _yolo_to_coco(\n    items: list[dict], names: list[str], destination: Path, image_ids: dict\n) -> dict:\n    from PIL import Image\n    images, annotations, annotation_id, skipped = [], [], 1, 0\n    class_instances: Counter = Counter()\n    for item in sorted(items, key=lambda v: image_ids[v["key"]]):\n        image_id = image_ids[item["key"]]\n        with Image.open(item["image"]) as handle:\n            width, height = handle.size\n        images.append({"id": image_id, "file_name": str(item["image"]),\n                       "width": width, "height": height})\n        for line in item["label"].read_text().splitlines():\n            values = line.strip().split()\n            if not values:\n                continue\n            if len(values) < 7 or (len(values) - 1) % 2:\n                skipped += 1; continue\n            try:\n                class_id = int(values[0])\n                coords = [float(v) for v in values[1:]]\n            except ValueError:\n                skipped += 1; continue\n            if not 0 <= class_id < len(names):\n                raise ValueError(f"Invalid class {class_id} in {item[\'label\']}")\n            xs = [max(0.0, min(1.0, coords[i])) * width for i in range(0, len(coords), 2)]\n            ys = [max(0.0, min(1.0, coords[i])) * height for i in range(1, len(coords), 2)]\n            area = _polygon_area(xs, ys)\n            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)\n            if area <= 1.0 or x1 <= x0 or y1 <= y0:\n                skipped += 1; continue\n            polygon = [c for pair in zip(xs, ys) for c in pair]\n            annotations.append({\n                "id": annotation_id, "image_id": image_id,\n                "category_id": class_id + 1,\n                "segmentation": [polygon], "area": float(area),\n                "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)],\n                "iscrowd": 0,\n            })\n            annotation_id += 1\n            class_instances[class_id] += 1\n    document = {\n        "images": images, "annotations": annotations,\n        "categories": [{"id": i + 1, "name": n, "supercategory": "skin"}\n                       for i, n in enumerate(names)],\n    }\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(document))\n    print(f"  [coco] {destination.name}: {len(images)} imgs, "\n          f"{len(annotations)} instances, skipped={skipped}")\n    return {\n        "images": len(images), "annotations": len(annotations),\n        "skipped_polygons": skipped,\n        "instances_per_class": {names[i]: int(class_instances[i]) for i in range(len(names))},\n    }\n\n\ndef _coco_evaluate(ground_truth: Path, predictions: Path, names: list[str]) -> dict:\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    detections = json.loads(predictions.read_text())\n    if not detections:\n        return {"bbox_mAP50": 0.0, "segm_mAP50": 0.0, "segm_mAP50_95": 0.0,\n                "per_class_mAP50": {name: 0.0 for name in names}}\n    coco_gt = COCO(str(ground_truth))\n    coco_dt = coco_gt.loadRes(str(predictions))\n    bbox = COCOeval(coco_gt, coco_dt, "bbox")\n    bbox.evaluate(); bbox.accumulate(); bbox.summarize()\n    segm = COCOeval(coco_gt, coco_dt, "segm")\n    segm.evaluate(); segm.accumulate(); segm.summarize()\n    per_class = {}\n    for cat_id, name in enumerate(names, start=1):\n        ev = COCOeval(coco_gt, coco_dt, "segm")\n        ev.params.catIds = [cat_id]\n        ev.evaluate(); ev.accumulate(); ev.summarize()\n        per_class[name] = float(ev.stats[1])\n    return {\n        "bbox_mAP50": float(bbox.stats[1]),\n        "segm_mAP50": float(segm.stats[1]),\n        "segm_mAP50_95": float(segm.stats[0]),\n        "per_class_mAP50": per_class,\n    }\n\n\ndef _merge_oof_predictions(files: list[Path], destination: Path) -> int:\n    merged = []\n    for path in files:\n        if not path.is_file():\n            raise FileNotFoundError(f"Missing OOF prediction file: {path}")\n        merged.extend(json.loads(path.read_text()))\n    destination.write_text(json.dumps(merged))\n    return len(merged)\n\n\ndef _balanced_subset(items: list[dict], limit: int, seed: int) -> list[dict]:\n    if len(items) <= limit:\n        return list(items)\n    shuffled = list(items)\n    random.Random(seed).shuffle(shuffled)\n    chosen, used = [], set()\n    for class_id in range(len(EXPECTED_CLASSES)):\n        for item in shuffled:\n            if item["key"] not in used and class_id in _item_classes(item):\n                chosen.append(item); used.add(item["key"]); break\n    for item in shuffled:\n        if len(chosen) >= limit: break\n        if item["key"] not in used:\n            chosen.append(item); used.add(item["key"])\n    return chosen\n\n\ndef _patch_mmdet_mmcv_guard() -> None:\n    """Allow the official Python-3.12 MMCV 2.2 wheel in MMDet 3.3."""\n    import importlib.util\n\n    spec = importlib.util.find_spec("mmdet")\n    if spec is None or spec.origin is None:\n        raise RuntimeError("mmdet was installed but its package path is missing")\n    init_path = Path(spec.origin)\n    source = init_path.read_text()\n    old = "mmcv_maximum_version = \'2.2.0\'"\n    if old in source:\n        init_path.write_text(source.replace(\n            old, "mmcv_maximum_version = \'2.3.0\'", 1\n        ))\n        print(f"  [deps] adjusted MMDetection\'s MMCV guard: {init_path}")\n\n\ndef _dependency_probe() -> str:\n    # Minimal probe: verify packages installed and CUDA ops importable.\n    # Does NOT import `from mmdet.models import CondInst` — that triggers the\n    # full model registry → seaborn → scipy chain in the subprocess.  The\n    # scipy chain fails when disk numpy was left in a damaged state by a prior\n    # broken session that ran `pip install numpy==1.26.4`.\n    code = (\n        "import torch, torchvision, mmcv, mmengine, mmdet; "\n        "from mmcv.ops import ModulatedDeformConv2dPack, nms; "\n        "assert torch.cuda.is_available(), \'CUDA unavailable\'; "\n        "print(torch.__version__, torchvision.__version__, "\n        "mmcv.__version__, mmengine.__version__, mmdet.__version__)"\n    )\n    result = subprocess.run(\n        [sys.executable, "-c", code],\n        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n    )\n    if result.returncode != 0:\n        raise subprocess.CalledProcessError(\n            result.returncode,\n            [sys.executable, "-c", "..."],\n            output=result.stdout,\n        )\n    return result.stdout.strip()\n\n\ndef _ensure_dependencies() -> None:\n    """Install a deterministic Kaggle Python-3.12 OpenMMLab stack."""\n    try:\n        print(f"  [deps] usable environment found: {_dependency_probe()}")\n        return\n    except (subprocess.CalledProcessError, OSError) as error:\n        detail = getattr(error, "output", "") or ""\n        if detail:\n            print("  [deps] current environment probe failed:")\n            print("\\n".join(detail.strip().splitlines()[-8:]))\n\n    print("  [deps] installing pinned Kaggle packages without openmim ...")\n    torch_info = subprocess.check_output(\n        [\n            sys.executable, "-c",\n            "import torch; print(torch.__version__.split(\'+\')[0]); "\n            "print(torch.version.cuda or \'cpu\')",\n        ],\n        text=True,\n    ).splitlines()\n    torch_version, cuda_version = torch_info[:2]\n    if not torch_version.startswith("2.4.") or not cuda_version.startswith("12.1"):\n        print(\n            f"  [deps] replacing torch {torch_version}/CUDA {cuda_version} "\n            "with torch 2.4/cu121"\n        )\n        _pip(\n            "--upgrade", "--force-reinstall", "--no-cache-dir",\n            "torch==2.4.0", "torchvision==0.19.0",\n            "--index-url", TORCH_INDEX,\n        )\n\n    # ── Do NOT downgrade numpy or scipy. ─────────────────────────────────────\n    # mmengine AND mmcv declare numpy<2.0 in their PyPI metadata; all OpenMMLab\n    # installs must use --no-deps so pip never touches numpy.\n    # numpy/scipy are intentionally omitted from the pre-install list; we will\n    # repair the disk state below after mmcv/mmdet are in place.\n    _pip(\n        "--upgrade",\n        "addict",                            # mmengine runtime dep\n        "termcolor",                         # mmengine runtime dep\n        "packaging",                         # mmengine runtime dep\n        "opencv-python-headless==4.10.0.84", # mmcv / mmdet image I/O\n        "matplotlib",\n        "pyyaml",\n        "Pillow",\n        "pycocotools",\n        "shapely",\n        "six",\n        "terminaltables",\n        "tqdm",\n    )\n    _pip("--upgrade", "--no-deps", "mmengine>=0.10.5,<1.0.0")\n    _pip(\n        "--upgrade", "--no-cache-dir", "--only-binary=mmcv", "--no-deps",\n        "mmcv==2.2.0", "-f", MMCV_INDEX,\n    )\n    _pip("--upgrade", "--no-deps", "mmdet==3.3.0")\n    _patch_mmdet_mmcv_guard()\n\n    # ── Repair disk numpy to >=2.0 AFTER all --no-deps installs. ─────────────\n    # A prior broken Kaggle run may have left numpy 1.26.4 on disk (from an\n    # older version of this script).  The running kernel always starts with\n    # numpy 2.5.x from the Kaggle container image, so upgrading disk files\n    # from 1.x → 2.x is safe: it brings the on-disk state back in sync with\n    # the in-memory C extension (2.5.x), eliminating the mixed state that\n    # causes `ImportError: _center` in fresh subprocesses.\n    _pip("--upgrade", "numpy>=2.0.0,<3.0.0")\n\n    # Verify in a fresh subprocess (clean import order, no mixed-state risk).\n    try:\n        print(f"  [deps] verified in a fresh interpreter: {_dependency_probe()}")\n    except subprocess.CalledProcessError as error:\n        output = getattr(error, "output", "") or "(no output)"\n        print("  [deps] post-install probe FAILED — full output:\\n" + output)\n        raise RuntimeError(\n            "OpenMMLab environment validation failed after installation. "\n            "See probe output above."\n        ) from error\n\n\n\n# %% 2. Dataset discovery and fold-integrity validation\ndef _split_dirs(root: Path, split: str) -> tuple[Path, Path]:\n    """Support Session-00 legacy and group-disjoint V2 directory layouts."""\n    legacy = (root / split / "images", root / split / "labels")\n    v2 = (root / "images" / split, root / "labels" / split)\n    if all(path.is_dir() for path in v2):\n        return v2\n    if all(path.is_dir() for path in legacy):\n        return legacy\n    return v2\n\n\ndef _valid_dataset_root(root: Path) -> bool:\n    if not (root / "data.yaml").is_file():\n        return False\n    # Only train and valid are mandatory. test is used for group-isolation\n    # checking when cv/ manifests are present, but is otherwise optional.\n    return all(\n        all(path.is_dir() for path in _split_dirs(root, split))\n        for split in ("train", "valid")\n    )\n\n\ndef _dataset_score(root: Path) -> tuple[int, int, int]:\n    return (\n        int(all((root / "cv" / f"fold_{fold}" / "train.txt").is_file()\n                and (root / "cv" / f"fold_{fold}" / "valid.txt").is_file()\n                for fold in range(5))),\n        int((root / "metadata" / "manifest.csv").is_file()),\n        int((root / "images" / "train").is_dir()),\n    )\n\n\ndef _find_dataset_root(hint: str) -> Path:\n    if hint != "auto":\n        explicit = Path(hint).expanduser().resolve()\n        if _valid_dataset_root(explicit):\n            print(f"  [data] explicit rebuilt dataset: {explicit}")\n            return explicit\n        raise FileNotFoundError(\n            f"--data-root is not a complete Session-00 dataset: {explicit}"\n        )\n\n    candidates: set[Path] = set()\n    search_roots = [Path("/kaggle/input")]\n    if not search_roots[0].is_dir():\n        search_roots.append(Path.cwd())\n    for search_root in search_roots:\n        if search_root.is_dir():\n            for data_yaml in search_root.rglob("data.yaml"):\n                root = data_yaml.parent.resolve()\n                if _valid_dataset_root(root):\n                    candidates.add(root)\n    for candidate in (\n        Path("/kaggle/working/Image_Annotation_rebuilt"),\n        Path("Image_Annotation_rebuilt"),\n        Path("../Image_Annotation_rebuilt"),\n    ):\n        if _valid_dataset_root(candidate):\n            candidates.add(candidate.resolve())\n    if not candidates:\n        raise FileNotFoundError(\n            "No complete rebuilt dataset was found. Attach the Session-00/V2 "\n            "dataset to Kaggle or set DATA_ROOT to its exact directory."\n        )\n\n    ranked = sorted(candidates, key=lambda path: (_dataset_score(path), str(path)),\n                    reverse=True)\n    best_score = _dataset_score(ranked[0])\n    tied = [path for path in ranked if _dataset_score(path) == best_score]\n    if len(tied) > 1:\n        choices = "\\n".join(f"    - {path}" for path in tied)\n        raise RuntimeError(\n            "Multiple equally suitable rebuilt datasets were found. Set "\n            f"DATA_ROOT explicitly:\\n{choices}"\n        )\n    print(f"  [data] selected rebuilt dataset: {ranked[0]}")\n    print(f"  [data] integrity features (cv, manifest, V2 layout)={best_score}")\n    return ranked[0]\n\n\ndef _load_class_names(data_yaml: Path) -> list[str]:\n    import yaml\n\n    document = yaml.safe_load(data_yaml.read_text()) or {}\n    names = document.get("names")\n    if isinstance(names, dict):\n        names = [names[key] for key in sorted(names, key=lambda value: int(value))]\n    names = [str(name) for name in names] if isinstance(names, list) else []\n    if names != EXPECTED_CLASSES:\n        raise ValueError(\n            f"Expected class order {EXPECTED_CLASSES}; data.yaml has {names}"\n        )\n    return names\n\n\ndef _gather_split(root: Path, split: str) -> list[dict]:\n    images_dir, labels_dir = _split_dirs(root, split)\n    items = []\n    for image in sorted(images_dir.iterdir()):\n        if image.suffix.lower() not in IMAGE_SUFFIXES:\n            continue\n        label = labels_dir / f"{image.stem}.txt"\n        if not label.is_file():\n            raise FileNotFoundError(f"Missing label for {image}: {label}")\n        items.append({\n            "image": image.resolve(),\n            "label": label.resolve(),\n            "key": str(image.resolve()),\n        })\n    if not items:\n        raise ValueError(f"No supported images found in {images_dir}")\n    return items\n\n\ndef _item_classes(item: dict) -> set[int]:\n    classes = set()\n    for line in item["label"].read_text().splitlines():\n        values = line.split()\n        if values:\n            classes.add(int(values[0]))\n    return classes\n\n\ndef _manifest_lookup(pool: list[dict]) -> dict[str, dict]:\n    lookup: dict[str, dict] = {}\n    for item in pool:\n        aliases = {item["key"], item["image"].name, item["image"].stem}\n        for alias in aliases:\n            if alias in lookup and lookup[alias]["key"] != item["key"]:\n                raise RuntimeError(\n                    f"Ambiguous image alias {alias}; fold manifests cannot be resolved"\n                )\n            lookup[alias] = item\n    return lookup\n\n\ndef _lookup_manifest_item(value: str, lookup: dict[str, dict]) -> dict | None:\n    candidate = Path(value)\n    return lookup.get(value) or lookup.get(candidate.name) or lookup.get(candidate.stem)\n\n\ndef _read_fold_list(\n    path: Path,\n    lookup: dict[str, dict],\n    excluded_lookup: dict[str, dict],\n) -> tuple[list[dict], int, int]:\n    resolved: dict[str, dict] = {}\n    missing = 0\n    excluded = 0\n    for raw in path.read_text().splitlines():\n        value = raw.strip()\n        if not value or value.startswith("#"):\n            continue\n        item = _lookup_manifest_item(value, lookup)\n        if item is None:\n            if _lookup_manifest_item(value, excluded_lookup) is not None:\n                excluded += 1\n            else:\n                missing += 1\n        else:\n            resolved[item["key"]] = item\n    return [resolved[key] for key in sorted(resolved)], missing, excluded\n\n\ndef _verify_group_manifest(root: Path, folds: list[dict], test: list[dict]) -> dict:\n    """Verify group isolation when the V2 metadata manifest is attached."""\n    manifest = root / "metadata" / "manifest.csv"\n    if not manifest.is_file():\n        return {\n            "group_manifest_present": False,\n            "group_disjoint_verified": False,\n            "reason": "metadata/manifest.csv is absent",\n        }\n    with manifest.open(newline="", encoding="utf-8") as handle:\n        rows = list(csv.DictReader(handle))\n    by_basename = {}\n    for row in rows:\n        image_ref = row.get("image_file", "")\n        if image_ref:\n            by_basename[Path(image_ref).name] = row\n    all_fold_items = [\n        item for current in folds for item in current["valid"]\n    ]\n    required_items = all_fold_items + test\n    missing_rows = [\n        item["image"].name for item in required_items\n        if item["image"].name not in by_basename\n    ]\n    missing_groups = [\n        item["image"].name for item in required_items\n        if item["image"].name in by_basename\n        and not by_basename[item["image"].name].get("split_group_id", "")\n    ]\n    if missing_rows or missing_groups:\n        raise RuntimeError(\n            "V2 group manifest is incomplete: "\n            f"missing_rows={len(missing_rows)}, missing_group_ids={len(missing_groups)}"\n        )\n    test_groups = {\n        by_basename[item["image"].name].get("split_group_id", "")\n        for item in test if item["image"].name in by_basename\n    }\n    test_groups.discard("")\n    fold_report = []\n    for current in folds:\n        train_groups = {\n            by_basename[item["image"].name].get("split_group_id", "")\n            for item in current["train"] if item["image"].name in by_basename\n        }\n        valid_groups = {\n            by_basename[item["image"].name].get("split_group_id", "")\n            for item in current["valid"] if item["image"].name in by_basename\n        }\n        train_groups.discard("")\n        valid_groups.discard("")\n        train_valid_overlap = train_groups & valid_groups\n        test_overlap = (train_groups | valid_groups) & test_groups\n        if train_valid_overlap or test_overlap:\n            raise RuntimeError(\n                f"Group leakage found in fold {current[\'fold\']}: "\n                f"train/valid={len(train_valid_overlap)}, development/test={len(test_overlap)}"\n            )\n        fold_report.append({\n            "fold": current["fold"],\n            "train_valid_group_overlap": 0,\n            "development_test_group_overlap": 0,\n        })\n    return {\n        "group_manifest_present": True,\n        "group_disjoint_verified": True,\n        "fold_checks": fold_report,\n    }\n\n\ndef _load_predefined_folds(\n    root: Path, development: list[dict], test: list[dict]\n) -> tuple[list[dict], dict] | None:\n    fold_dirs = [root / "cv" / f"fold_{fold}" for fold in range(5)]\n    if not all((folder / "train.txt").is_file()\n               and (folder / "valid.txt").is_file() for folder in fold_dirs):\n        return None\n    lookup = _manifest_lookup(development)\n    test_lookup = _manifest_lookup(test)\n    development_keys = {item["key"] for item in development}\n    validation_counts: Counter = Counter()\n    folds = []\n    stale = 0\n    excluded_test = 0\n    for fold, folder in enumerate(fold_dirs):\n        train, train_missing, train_test = _read_fold_list(\n            folder / "train.txt", lookup, test_lookup\n        )\n        valid, valid_missing, valid_test = _read_fold_list(\n            folder / "valid.txt", lookup, test_lookup\n        )\n        stale += train_missing + valid_missing\n        excluded_test += train_test + valid_test\n        train_keys = {item["key"] for item in train}\n        valid_keys = {item["key"] for item in valid}\n        if train_keys & valid_keys:\n            raise RuntimeError(f"Fold {fold} has train/validation image overlap")\n        if train_keys | valid_keys != development_keys:\n            absent = development_keys - (train_keys | valid_keys)\n            raise RuntimeError(\n                f"Fold {fold} does not cover {len(absent)} available development images"\n            )\n        validation_counts.update(valid_keys)\n        folds.append({"fold": fold, "train": train, "valid": valid})\n    bad = {key: count for key, count in validation_counts.items() if count != 1}\n    if set(validation_counts) != development_keys or bad:\n        raise RuntimeError(\n            "Five-fold validation must cover every development image exactly once; "\n            f"bad_count={len(bad)}"\n        )\n    group_check = _verify_group_manifest(root, folds, test)\n    integrity = {\n        "cv_mode": "predefined Session-00 five-fold manifests",\n        "development_images": len(development),\n        "untouched_test_images": len(test),\n        "fold_val_sizes": {item["fold"]: len(item["valid"]) for item in folds},\n        "test_manifest_references_excluded": excluded_test,\n        "missing_manifest_references_filtered": stale,\n        **group_check,\n    }\n    return folds, integrity\n\n\ndef _multi_hot(items: list[dict], num_classes: int) -> "np.ndarray":\n    import numpy as np\n\n    matrix = np.zeros((len(items), num_classes), dtype=np.int32)\n    for index, item in enumerate(items):\n        for class_id in _item_classes(item):\n            if 0 <= class_id < num_classes:\n                matrix[index, class_id] = 1\n    return matrix\n\n\ndef _fallback_folds(\n    development: list[dict], num_classes: int, seed: int\n) -> tuple[list[dict], dict]:\n    """Image-level fallback only; V2 predefined group folds are preferred.\n\n    Pure-numpy implementation — no sklearn/scipy dependency.\n    sklearn -> scipy -> numpy version-mismatch crashes the kernel (ImportError:\n    _center from \'numpy._core.umath\').\n    """\n    import numpy as np\n\n    labels = _multi_hot(development, num_classes)\n    dominant = labels.argmax(axis=1)\n\n    # --- pure-numpy stratified k-fold -----------------------------------\n    rng = np.random.RandomState(seed)\n    n = len(development)\n    n_splits = 5\n    fold_of = np.full(n, -1, dtype=np.int32)\n    classes = np.unique(dominant)\n    for cls in classes:\n        idx = np.where(dominant == cls)[0]\n        rng.shuffle(idx)\n        for k, chunk in enumerate(np.array_split(idx, n_splits)):\n            for i in chunk:\n                if fold_of[i] == -1:\n                    fold_of[i] = k % n_splits\n    # assign any still-unassigned (edge case: empty dominant class)\n    unassigned = np.where(fold_of == -1)[0]\n    if len(unassigned):\n        for k, i in enumerate(unassigned):\n            fold_of[i] = k % n_splits\n    # -----------------------------------------------------------------------\n\n    folds = []\n    for fold in range(n_splits):\n        folds.append({\n            "fold": fold,\n            "train": [item for index, item in enumerate(development)\n                      if fold_of[index] != fold],\n            "valid": [item for index, item in enumerate(development)\n                      if fold_of[index] == fold],\n        })\n    return folds, {\n        "cv_mode": "image-level stratified fallback",\n        "development_images": len(development),\n        "group_manifest_present": False,\n        "group_disjoint_verified": False,\n        "warning": (\n            "No predefined cv manifests were found. These folds are image-disjoint "\n            "but cannot prove patient/case/source-group isolation."\n        ),\n        "fold_val_sizes": {item["fold"]: len(item["valid"]) for item in folds},\n    }\n\n\n# %% 3. Model configuration and official checkpoint adaptation\ndef _detect_gpu_ids(device: str) -> list[str]:\n    if device != "auto":\n        identifiers = [part.strip() for part in device.split(",") if part.strip()]\n        if device == "cpu" or not identifiers:\n            raise RuntimeError("Session 24 requires a CUDA GPU")\n        return identifiers\n    count = int(subprocess.check_output(\n        [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],\n        text=True,\n    ).strip())\n    if count < 1:\n        raise RuntimeError("No CUDA GPU detected. Enable a Kaggle GPU accelerator.")\n    return [str(index) for index in range(min(2, count))]\n\n\ndef _write_training_config(\n    destination: Path,\n    train_json: Path,\n    valid_json: Path,\n    work_dir: Path,\n    checkpoint: Path,\n    names: list[str],\n    gpu_count: int,\n    max_iters: int,\n    image_size: int,\n    accumulation: int,\n    seed: int,\n) -> None:\n    classes = repr(tuple(names))\n    palette = repr([(220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230)])\n    interval = max(20 if max_iters <= 100 else 500, max_iters // 4)\n    warmup = min(500, max(10, max_iters // 10))\n    effective_batch = gpu_count * accumulation  # batch_per_gpu is fixed at one\n    learning_rate = 0.01 * effective_batch / 16.0\n    milestone_1 = max(warmup + 1, int(max_iters * 0.80))\n    milestone_2 = max(milestone_1 + 1, int(max_iters * 0.92))\n    config = f\'\'\'# Auto-generated by SESSION 24.\n_base_ = [\'mmdet::condinst/condinst_r50_fpn_ms-poly-90k_coco_instance.py\']\n\ndefault_scope = \'mmdet\'\nmetainfo = dict(classes={classes}, palette={palette})\nimage_size = ({image_size}, {image_size})\n\nmodel = dict(\n    backbone=dict(\n        depth=101,\n        frozen_stages=1,\n        with_cp=True,\n        init_cfg=dict(type=\'Pretrained\', checkpoint=\'torchvision://resnet101\'),\n        # Genuine DCNv2 in all ResNet-101 bottlenecks of C3, C4 and C5.\n        dcn=dict(type=\'DCNv2\', deform_groups=1, fallback_on_stride=False),\n        stage_with_dcn=(False, True, True, True),\n    ),\n    bbox_head=dict(num_classes={len(names)}),\n    mask_head=dict(max_masks_to_train=150),\n)\n\ntrain_pipeline = [\n    dict(type=\'LoadImageFromFile\', to_float32=True),\n    dict(type=\'LoadAnnotations\', with_bbox=True, with_mask=True, poly2mask=True),\n    dict(type=\'RandomFlip\', prob=0.5),\n    dict(type=\'RandomResize\', scale=image_size, ratio_range=(0.75, 1.25),\n         resize_type=\'Resize\', keep_ratio=True),\n    dict(type=\'RandomCrop\', crop_size=image_size, crop_type=\'absolute\',\n         recompute_bbox=True, allow_negative_crop=True),\n    dict(type=\'FilterAnnotations\', min_gt_bbox_wh=(1.0, 1.0), by_mask=True),\n    dict(type=\'Pad\', size=image_size),\n    dict(type=\'PackDetInputs\'),\n]\ntest_pipeline = [\n    dict(type=\'LoadImageFromFile\', to_float32=True),\n    dict(type=\'Resize\', scale=image_size, keep_ratio=True),\n    dict(type=\'Pad\', size=image_size),\n    dict(type=\'PackDetInputs\', meta_keys=(\n        \'img_id\', \'img_path\', \'ori_shape\', \'img_shape\', \'scale_factor\')),\n]\n\ntrain_dataloader = dict(\n    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,\n    sampler=dict(type=\'InfiniteSampler\', shuffle=True),\n    dataset=dict(\n        type=\'CocoDataset\', data_root=\'\', ann_file={str(train_json)!r},\n        data_prefix=dict(img=\'\'), metainfo=metainfo,\n        filter_cfg=dict(filter_empty_gt=True, min_size=1),\n        pipeline=train_pipeline,\n    ),\n)\nval_dataloader = dict(\n    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,\n    drop_last=False, sampler=dict(type=\'DefaultSampler\', shuffle=False),\n    dataset=dict(\n        type=\'CocoDataset\', data_root=\'\', ann_file={str(valid_json)!r},\n        data_prefix=dict(img=\'\'), metainfo=metainfo, test_mode=True,\n        pipeline=test_pipeline,\n    ),\n)\ntest_dataloader = val_dataloader\nval_evaluator = dict(\n    _delete_=True, type=\'CocoMetric\', ann_file={str(valid_json)!r},\n    metric=[\'bbox\', \'segm\'], classwise=True,\n)\ntest_evaluator = val_evaluator\n\noptim_wrapper = dict(\n    _delete_=True, type=\'AmpOptimWrapper\', loss_scale=\'dynamic\',\n    accumulative_counts={accumulation},\n    optimizer=dict(type=\'SGD\', lr={learning_rate}, momentum=0.9,\n                   weight_decay=0.0001),\n    clip_grad=dict(max_norm=35.0, norm_type=2),\n)\nparam_scheduler = [\n    dict(type=\'LinearLR\', start_factor=0.001, by_epoch=False,\n         begin=0, end={warmup}),\n    dict(type=\'MultiStepLR\', begin=0, end={max_iters}, by_epoch=False,\n         milestones=[{milestone_1}, {milestone_2}], gamma=0.1),\n]\ntrain_cfg = dict(\n    _delete_=True, type=\'IterBasedTrainLoop\', max_iters={max_iters},\n    val_interval={interval},\n)\nval_cfg = dict(_delete_=True, type=\'ValLoop\')\ntest_cfg = dict(_delete_=True, type=\'TestLoop\')\ndefault_hooks = dict(\n    logger=dict(type=\'LoggerHook\', interval=50),\n    checkpoint=dict(\n        type=\'CheckpointHook\', by_epoch=False, interval={interval},\n        save_last=True, save_best=\'coco/segm_mAP_50\', rule=\'greater\',\n        max_keep_ckpts=2,\n    ),\n)\nlog_processor = dict(type=\'LogProcessor\', window_size=50, by_epoch=False)\nrandomness = dict(seed={seed}, deterministic=False, diff_rank_seed=True)\nenv_cfg = dict(\n    cudnn_benchmark=True,\n    mp_cfg=dict(mp_start_method=\'fork\', opencv_num_threads=0),\n    dist_cfg=dict(backend=\'nccl\'),\n)\nlauncher = \'none\'\nwork_dir = {str(work_dir)!r}\nload_from = {str(checkpoint)!r}\nresume = False\n\'\'\'\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(config)\n\n\ndef _validate_model_build(config: Path) -> None:\n    """Build in a fresh process and prove CondInst/R101/DCNv2 are genuine."""\n    code = r\'\'\'\nimport sys\nfrom mmengine.config import Config\nfrom mmdet.registry import MODELS\nfrom mmdet.utils import register_all_modules\nfrom mmcv.ops import ModulatedDeformConv2dPack\nregister_all_modules(init_default_scope=True)\ncfg = Config.fromfile(sys.argv[1])\nmodel = MODELS.build(cfg.model)\nassert model.__class__.__name__ == \'CondInst\'\nassert model.backbone.__class__.__name__ == \'ResNet\'\nassert model.backbone.depth == 101\nassert tuple(model.backbone.stage_blocks) == (3, 4, 23, 3)\nassert tuple(model.backbone.stage_with_dcn) == (False, True, True, True)\nassert model.bbox_head.__class__.__name__ == \'CondInstBboxHead\'\nassert model.mask_head.__class__.__name__ == \'CondInstMaskHead\'\nassert model.bbox_head.num_params == 169\nassert model.mask_head.num_params == 169\ndcn = sum(isinstance(module, ModulatedDeformConv2dPack)\n          for module in model.modules())\nassert dcn == 30, f\'expected 30 DCNv2 bottlenecks in C3-C5, found {dcn}\'\nprint(\'architecture=CondInst; backbone=ResNet-101; \'\n      \'DCNv2_C3-C5=30; dynamic_mask_parameters=169\')\n\'\'\'\n    subprocess.check_call([sys.executable, "-c", code, str(config)])\n\n\ndef _prepare_condinst_checkpoint(work: Path, config: Path) -> tuple[Path, dict]:\n    """Reuse compatible official CondInst heads while R101 initializes from ImageNet."""\n    prepared = work / "pretrained" / "condinst_r50_heads_for_r101_dcn.pth"\n    report_path = work / "pretrained" / "checkpoint_adaptation.json"\n    if prepared.is_file() and prepared.stat().st_size > 1_000_000 \\\n            and report_path.is_file():\n        print(f"  [weights] prepared checkpoint exists: {prepared}")\n        return prepared, json.loads(report_path.read_text())\n\n    raw = work / "pretrained" / "condinst_r50_coco_raw.pth"\n    raw.parent.mkdir(parents=True, exist_ok=True)\n    print("  [weights] downloading official MMDetection CondInst-R50 COCO weights ...")\n    try:\n        urlretrieve(CONDINST_R50_URL, raw)\n    except Exception as error:\n        raw.unlink(missing_ok=True)\n        raise RuntimeError(\n            "Official CondInst checkpoint download failed. Enable Kaggle Internet "\n            "and rerun. No fallback model will be used."\n        ) from error\n    if raw.stat().st_size <= 1_000_000:\n        raw.unlink(missing_ok=True)\n        raise RuntimeError("Downloaded CondInst checkpoint is unexpectedly small")\n\n    import torch\n    from mmengine.config import Config\n    from mmdet.registry import MODELS\n    from mmdet.utils import register_all_modules\n\n    register_all_modules(init_default_scope=True)\n    cfg = Config.fromfile(str(config))\n    model = MODELS.build(cfg.model)\n    target = model.state_dict()\n    checkpoint = torch.load(raw, map_location="cpu", weights_only=False)\n    loaded = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))\n    source = {\n        (key[7:] if key.startswith("module.") else key): value\n        for key, value in loaded.items()\n    }\n    retained = {}\n    rejected_backbone = []\n    rejected_shape = []\n    rejected_missing = []\n    for key, value in source.items():\n        if key.startswith("backbone."):\n            rejected_backbone.append(key)\n        elif key not in target:\n            rejected_missing.append(key)\n        elif tuple(value.shape) != tuple(target[key].shape):\n            rejected_shape.append(key)\n        else:\n            retained[key] = value\n    prefixes = {\n        "neck": any(key.startswith("neck.") for key in retained),\n        "bbox_head": any(key.startswith("bbox_head.") for key in retained),\n        "mask_head": any(key.startswith("mask_head.") for key in retained),\n        "dynamic_controller": any("bbox_head.controller" in key for key in retained),\n    }\n    if not all(prefixes.values()) or len(retained) < 50:\n        raise RuntimeError(\n            "Official checkpoint did not contain the expected compatible CondInst "\n            f"neck/dynamic heads: {prefixes}; retained={len(retained)}"\n        )\n    adapted = {\n        "state_dict": retained,\n        "meta": {\n            "source": CONDINST_R50_URL,\n            "adaptation": (\n                "Removed R50 backbone; retained only exact-shape FPN and CondInst "\n                "head tensors. R101-DCNv2 uses torchvision ImageNet initialization."\n            ),\n        },\n    }\n    torch.save(adapted, prepared)\n    raw.unlink(missing_ok=True)\n    report = {\n        "source_url": CONDINST_R50_URL,\n        "retained_tensors": len(retained),\n        "removed_r50_backbone_tensors": len(rejected_backbone),\n        "removed_shape_mismatches": rejected_shape,\n        "removed_unknown_tensors": len(rejected_missing),\n        "required_prefixes_present": prefixes,\n    }\n    report_path.write_text(json.dumps(report, indent=2))\n    print(\n        f"  [weights] retained={len(retained)} exact-shape FPN/head tensors; "\n        f"removed_backbone={len(rejected_backbone)}; "\n        f"shape_mismatch={len(rejected_shape)}"\n    )\n    return prepared, report\n\n\ndef _write_prediction_config(\n    train_config: Path, destination: Path, valid_json: Path, prefix: Path\n) -> None:\n    override = f\'\'\'\n\n# Strict out-of-fold prediction override generated by SESSION 24.\ntest_dataloader = dict(\n    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,\n    drop_last=False, sampler=dict(type=\'DefaultSampler\', shuffle=False),\n    dataset=dict(\n        type=\'CocoDataset\', data_root=\'\', ann_file={str(valid_json)!r},\n        data_prefix=dict(img=\'\'), metainfo=metainfo, test_mode=True,\n        pipeline=test_pipeline,\n    ),\n)\ntest_evaluator = dict(\n    _delete_=True, type=\'CocoMetric\', ann_file={str(valid_json)!r},\n    metric=[\'bbox\', \'segm\'], format_only=True,\n    outfile_prefix={str(prefix)!r},\n)\nload_from = None\nresume = False\n\'\'\'\n    destination.write_text(train_config.read_text() + override)\n\n\n# %% 4. Resume-safe completion markers\ndef _completed_checkpoint(work_dir: Path, config: Path) -> Path | None:\n    marker = work_dir / "SESSION_24_TRAINING_COMPLETE.json"\n    if not marker.is_file():\n        return None\n    try:\n        record = json.loads(marker.read_text())\n        digest = hashlib.sha256(config.read_bytes()).hexdigest()\n        checkpoint = Path(record["checkpoint"])\n        if record.get("config_sha256") == digest and checkpoint.is_file():\n            return checkpoint\n    except (KeyError, OSError, ValueError, json.JSONDecodeError):\n        pass\n    return None\n\n\ndef _mark_training_complete(work_dir: Path, config: Path, checkpoint: Path) -> None:\n    marker = work_dir / "SESSION_24_TRAINING_COMPLETE.json"\n    marker.write_text(json.dumps({\n        "checkpoint": str(checkpoint),\n        "config_sha256": hashlib.sha256(config.read_bytes()).hexdigest(),\n        "completed_utc": datetime.now(timezone.utc).isoformat(),\n    }, indent=2))\n\n\n# %% 5. End-to-end Session 24 run\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(\n        description="SESSION 24: CondInst ResNet-101 DCNv2 five-fold OOF CV"\n    )\n    parser.add_argument("--data-root", default="auto")\n    parser.add_argument("--work-dir", default=None)\n    parser.add_argument("--device", default="auto", help="auto, 0, or 0,1")\n    parser.add_argument("--folds", default="0,1,2,3,4")\n    parser.add_argument("--max-iters", type=int, default=3000,\n                        help="Iters per fold. 5 folds x 3000 ~10h on T4x2. Fits 12h.")\n    parser.add_argument("--image-size", type=int, default=640)\n    parser.add_argument("--accumulation", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--quick", action="store_true")\n    parser.add_argument("--skip-existing", action="store_true", default=True)\n    parser.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")\n    parser.add_argument("--resume", action="store_true")\n    parser.add_argument("--skip-train", action="store_true")\n    args, _ = parser.parse_known_args(argv)\n\n    if args.max_iters < 1:\n        raise ValueError("--max-iters must be positive")\n    if args.image_size < 384 or args.image_size % 32:\n        raise ValueError("--image-size must be >=384 and divisible by 32")\n    if args.accumulation < 1:\n        raise ValueError("--accumulation must be positive")\n\n    multiprocessing.freeze_support()\n    started = time.time()\n    print("=" * 76)\n    print("SESSION 24 — genuine CondInst R101-DCNv2 + five-fold OOF CV")\n    print("=" * 76)\n    _ensure_dependencies()\n    gpu_ids = _detect_gpu_ids(args.device)\n    selected_folds = _parse_folds(args.folds)\n    print(f"  [gpu] training devices={gpu_ids}; folds={selected_folds}")\n\n    root = _find_dataset_root(args.data_root)\n    names = _load_class_names(root / "data.yaml")\n    train_items = _gather_split(root, "train")\n    valid_items = _gather_split(root, "valid")\n    # test split is optional — used for group-isolation checks when cv/ manifests\n    # are present. Absent test/ is silently treated as an empty holdout set.\n    try:\n        test_items = _gather_split(root, "test")\n    except (FileNotFoundError, ValueError):\n        test_items = []\n    development = train_items + valid_items\n    if len({item["key"] for item in development}) != len(development):\n        raise RuntimeError("Duplicate image paths exist in the development pool")\n    print(\n        f"  [data] development={len(development)} "\n        f"({len(train_items)} train + {len(valid_items)} valid); "\n        f"untouched_test={len(test_items)}"\n    )\n    predefined = _load_predefined_folds(root, development, test_items)\n    if predefined is None:\n        fold_data, integrity = _fallback_folds(development, len(names), args.seed)\n        print("  [integrity] WARNING: using image-level fallback folds")\n    else:\n        fold_data, integrity = predefined\n        print(\n            "  [integrity] predefined five-fold manifests validated; "\n            f"group_disjoint_verified={integrity[\'group_disjoint_verified\']}"\n        )\n\n    default_work = Path("/kaggle/working") / (\n        f"{SESSION_NAME}_quick" if args.quick else SESSION_NAME\n    )\n    work = Path(args.work_dir).expanduser().resolve() if args.work_dir else default_work\n    work.mkdir(parents=True, exist_ok=True)\n    coco_dir = work / "coco"\n    configs_dir = work / "configs"\n    predictions_dir = work / "oof_predictions"\n    for directory in (coco_dir, configs_dir, predictions_dir):\n        directory.mkdir(parents=True, exist_ok=True)\n\n    if args.quick:\n        selected_folds = [selected_folds[0]]\n        args.max_iters = min(args.max_iters, 20)\n        print("  [quick] one fold, <=48 train / <=24 validation images, 20 iterations")\n    selected_data = []\n    for fold in selected_folds:\n        current = dict(fold_data[fold])\n        if args.quick:\n            current["train"] = _balanced_subset(current["train"], 48, args.seed)\n            current["valid"] = _balanced_subset(current["valid"], 24, args.seed + 1)\n        selected_data.append(current)\n\n    all_items_by_key = {\n        item["key"]: item\n        for current in selected_data\n        for item in current["train"] + current["valid"]\n    }\n    image_ids = {\n        key: index + 1 for index, key in enumerate(sorted(all_items_by_key))\n    }\n    oof_items_by_key = {\n        item["key"]: item for current in selected_data for item in current["valid"]\n    }\n    oof_items = [oof_items_by_key[key] for key in sorted(oof_items_by_key)]\n    oof_gt = coco_dir / "selected_oof_ground_truth.json"\n    oof_conversion = _yolo_to_coco(oof_items, names, oof_gt, image_ids)\n\n    # Write a representative config before adapting the official checkpoint.\n    bootstrap_config = configs_dir / "architecture_preflight.py"\n    prepared_path = work / "pretrained" / "condinst_r50_heads_for_r101_dcn.pth"\n    _write_training_config(\n        bootstrap_config, oof_gt, oof_gt, work / "preflight", prepared_path,\n        names, len(gpu_ids), args.max_iters, args.image_size,\n        args.accumulation, args.seed,\n    )\n    compile(bootstrap_config.read_text(), str(bootstrap_config), "exec")\n    print("  [preflight] building model and verifying CondInst/R101/DCNv2 ...")\n    _validate_model_build(bootstrap_config)\n    pretrained, adaptation_report = _prepare_condinst_checkpoint(\n        work, bootstrap_config\n    )\n\n    summary = {\n        "session": "SESSION_24",\n        "model": "CondInst with ResNet-101 and DCNv2 in C3-C5",\n        "method": (\n            "five-fold cross-validation with strict out-of-fold evaluation; "\n            "CondInst per-instance dynamic convolution masks"\n        ),\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "dataset_root": str(root),\n        "classes": names,\n        "selected_folds": selected_folds,\n        "dataset_integrity": integrity,\n        "oof_conversion": oof_conversion,\n        "checkpoint_adaptation": adaptation_report,\n        "configuration": {\n            "max_iters_per_fold": args.max_iters,\n            "image_size": args.image_size,\n            "gpus": gpu_ids,\n            "batch_per_gpu": 1,\n            "gradient_accumulation": args.accumulation,\n            "quick": args.quick,\n            "backbone": "ResNet-101",\n            "dcn_type": "DCNv2",\n            "dcn_stages": ["C3", "C4", "C5"],\n            "verified_dcn_layers": 30,\n            "dynamic_mask_parameters_per_instance": 169,\n        },\n        "fold_results": [],\n        "oof": {},\n        "limitations": [\n            "The 0.73-0.78 mAP50 range in model_strategy_analysis.md is a "\n            "planning estimate, not a measured result or guarantee.",\n            "The untouched test split is not evaluated during CV; use it once "\n            "after model selection for a final locked estimate.",\n            "Group-disjoint claims are made only when metadata/manifest.csv is "\n            "present and its zero-overlap checks pass.",\n            "MMDetection 3.3 with the Python-3.12 MMCV 2.2 wheel uses a narrowly "\n            "scoped version-guard compatibility adjustment.",\n        ],\n    }\n    summary_path = work / "SESSION_24_results.json"\n\n    def save_summary() -> None:\n        summary_path.write_text(json.dumps(summary, indent=2, default=str))\n\n    prediction_files = []\n    for current in selected_data:\n        fold = current["fold"]\n        print("\\n" + "-" * 76)\n        print(f"FOLD {fold}: train={len(current[\'train\'])}, valid={len(current[\'valid\'])}")\n        print("-" * 76)\n        fold_coco = coco_dir / f"fold_{fold}"\n        fold_config_dir = configs_dir / f"fold_{fold}"\n        fold_work = work / "folds" / f"fold_{fold}"\n        for directory in (fold_coco, fold_config_dir, fold_work):\n            directory.mkdir(parents=True, exist_ok=True)\n        train_json = fold_coco / "train.json"\n        valid_json = fold_coco / "valid.json"\n        train_report = _yolo_to_coco(current["train"], names, train_json, image_ids)\n        valid_report = _yolo_to_coco(current["valid"], names, valid_json, image_ids)\n        config = fold_config_dir / "condinst_r101_dcn.py"\n        _write_training_config(\n            config, train_json, valid_json, fold_work, pretrained, names,\n            len(gpu_ids), args.max_iters, args.image_size, args.accumulation,\n            args.seed + fold,\n        )\n        compile(config.read_text(), str(config), "exec")\n\n        checkpoint = _best_checkpoint(fold_work)\n        completed_checkpoint = _completed_checkpoint(fold_work, config)\n        if args.skip_train:\n            if checkpoint is None:\n                raise FileNotFoundError(\n                    f"--skip-train was set but fold {fold} has no checkpoint"\n                )\n        elif args.skip_existing and completed_checkpoint is not None:\n            checkpoint = completed_checkpoint\n            print(f"  [train] reusing completed fold checkpoint: {checkpoint}")\n        else:\n            if checkpoint is not None and not args.resume:\n                print(\n                    "  [train] checkpoint exists without a matching completion "\n                    "marker; starting this fold from iteration 0. Set RESUME=True "\n                    "to continue an interrupted fold."\n                )\n            _run_training(config, fold_work, gpu_ids, args.resume)\n            checkpoint = _best_checkpoint(fold_work)\n            if checkpoint is None:\n                raise RuntimeError(f"Fold {fold} finished without a checkpoint")\n            _mark_training_complete(fold_work, config, checkpoint)\n\n        prediction_prefix = predictions_dir / f"fold_{fold}_valid"\n        prediction_config = fold_config_dir / "predict_valid.py"\n        _write_prediction_config(\n            config, prediction_config, valid_json, prediction_prefix\n        )\n        print(f"  [oof] predicting fold {fold} validation with {checkpoint.name}")\n        _run_prediction(prediction_config, checkpoint, fold_work, gpu_ids[0])\n        prediction_json = Path(f"{prediction_prefix}.segm.json")\n        if not prediction_json.is_file():\n            raise FileNotFoundError(f"MMDetection did not create {prediction_json}")\n        metrics = _coco_evaluate(valid_json, prediction_json, names)\n        prediction_files.append(prediction_json)\n        summary["fold_results"].append({\n            "fold": fold,\n            "checkpoint": str(checkpoint),\n            "train_conversion": train_report,\n            "valid_conversion": valid_report,\n            "metrics": metrics,\n        })\n        save_summary()\n\n    oof_predictions = predictions_dir / "SESSION_24_oof.segm.json"\n    detection_count = _merge_oof_predictions(prediction_files, oof_predictions)\n    oof_metrics = _coco_evaluate(oof_gt, oof_predictions, names)\n    fold_scores = [\n        result["metrics"]["segm_mAP50"] for result in summary["fold_results"]\n    ]\n    summary["oof"] = {\n        "evaluated_images": len(oof_items),\n        "detections": detection_count,\n        "metrics": oof_metrics,\n        "fold_mean_segm_mAP50": float(statistics.mean(fold_scores)),\n        "fold_std_segm_mAP50": (\n            float(statistics.pstdev(fold_scores)) if len(fold_scores) > 1 else 0.0\n        ),\n        "complete_five_fold_oof": selected_folds == [0, 1, 2, 3, 4]\n        and not args.quick,\n    }\n    summary["elapsed_hours"] = (time.time() - started) / 3600.0\n    save_summary()\n\n    archive = Path("/kaggle/working/SESSION_24_results.zip")\n    if not Path("/kaggle/working").is_dir():\n        archive = work / "SESSION_24_results.zip"\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output_zip:\n        output_zip.write(summary_path, arcname=summary_path.name)\n        output_zip.write(oof_gt, arcname=oof_gt.name)\n        output_zip.write(oof_predictions, arcname=oof_predictions.name)\n        adaptation_path = work / "pretrained" / "checkpoint_adaptation.json"\n        output_zip.write(adaptation_path, arcname=adaptation_path.name)\n        for current, prediction in zip(selected_data, prediction_files):\n            fold = current["fold"]\n            config = configs_dir / f"fold_{fold}" / "condinst_r101_dcn.py"\n            output_zip.write(config, arcname=f"fold_{fold}/{config.name}")\n            output_zip.write(prediction, arcname=f"fold_{fold}/{prediction.name}")\n\n    print("\\n" + "=" * 76)\n    print("SESSION 24 COMPLETE")\n    print("=" * 76)\n    print(\n        f"OOF segm mAP50={oof_metrics[\'segm_mAP50\']:.4f}; "\n        f"mAP50-95={oof_metrics[\'segm_mAP50_95\']:.4f}; "\n        f"bbox mAP50={oof_metrics[\'bbox_mAP50\']:.4f}"\n    )\n    print("Per-class OOF segmentation mAP50:")\n    for name, value in oof_metrics["per_class_mAP50"].items():\n        print(f"  {name}: {value:.4f}")\n    print(f"Results JSON: {summary_path}")\n    print(f"Results ZIP : {archive}")\n    print(f"Checkpoints : {work / \'folds\'}  (download separately)")\n    return 0\n\n\n', _SESSION_24_condinst_r101_dcn_cv_module.__file__, 'exec'), _SESSION_24_condinst_r101_dcn_cv_module.__dict__)

_KAGGLE_DATASET_REBUILD_V2_module = types.ModuleType('KAGGLE_DATASET_REBUILD_V2')
_KAGGLE_DATASET_REBUILD_V2_module.__file__ = '<embedded-KAGGLE_DATASET_REBUILD_V2>'
sys.modules['KAGGLE_DATASET_REBUILD_V2'] = _KAGGLE_DATASET_REBUILD_V2_module
exec(compile('#!/usr/bin/env python3\n"""Kaggle-ready, group-disjoint rebuild for the skin-disease image dataset.\n\nThe script has two honest operating modes:\n\n* ``coarse`` uses one canonical image and the existing quadrilateral labels per\n  inferred source photograph.  Its output is suitable for coarse localization\n  or weak-supervision experiments, not detailed lesion-boundary claims.\n* ``reviewed`` uses approved replacement YOLO polygon labels from a review\n  bundle.  It exports only approved cases and is the path for detailed-mask\n  experiments.\n\nIt never invents patient IDs, clinical masks, provenance, or reviewer approval.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport itertools\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport sys\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageOps\n\n\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\nEXPECTED_CLASSES = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]\nCOLORS = [(255, 55, 55), (45, 205, 80), (60, 145, 255), (255, 185, 25)]\nIGNORE_INDEX = 255\n\n# Fixed offline transformations visible in this export.  This deliberately\n# removes only a known prefix and preserves the remaining source/export suffix.\nAUGMENT_PREFIX = re.compile(\n    r"^(?:"\n    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"\n    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"\n    r"Resize-450_400|"\n    r"flip-(?:-1|0|1)|"\n    r"(?:hue|saturation)-(?:200|150|100|50)|"\n    r"superpixels-(?:200|100|75|50|25)"\n    r")"\n)\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef stable_id(prefix: str, value: str, width: int = 16) -> str:\n    return prefix + hashlib.sha256(value.encode("utf-8")).hexdigest()[:width]\n\n\ndef inferred_source_key(stem: str) -> str:\n    before_roboflow = stem.split(".rf.", 1)[0]\n    return AUGMENT_PREFIX.sub("", before_roboflow, count=1)\n\n\ndef is_augmented(stem: str) -> bool:\n    return AUGMENT_PREFIX.match(stem.split(".rf.", 1)[0]) is not None\n\n\ndef write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str] | None = None) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if fieldnames is None:\n        fieldnames = list(rows[0]) if rows else []\n    with path.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef read_csv(path: Path | None) -> list[dict[str, str]]:\n    if path is None:\n        return []\n    with path.open(newline="", encoding="utf-8-sig") as stream:\n        return list(csv.DictReader(stream))\n\n\ndef find_dataset_root(explicit: str | None) -> Path:\n    if explicit:\n        candidate = Path(explicit).expanduser().resolve()\n        if not candidate.is_dir():\n            raise FileNotFoundError(f"Input dataset directory does not exist: {candidate}")\n        validate_dataset_layout(candidate)\n        return candidate\n\n    search_root = Path("/kaggle/input")\n    candidates: list[Path] = []\n    if search_root.is_dir():\n        for directory in search_root.rglob("Image_Annotation_rebuilt"):\n            if directory.is_dir() and has_dataset_layout(directory):\n                candidates.append(directory.resolve())\n        if not candidates:\n            for yaml_path in search_root.rglob("data.yaml"):\n                if has_dataset_layout(yaml_path.parent):\n                    candidates.append(yaml_path.parent.resolve())\n    candidates = sorted(set(candidates))\n    if len(candidates) != 1:\n        shown = "\\n".join(f"  - {p}" for p in candidates) or "  (none)"\n        raise RuntimeError(\n            "Expected exactly one uploaded rebuilt dataset. Set INPUT_ROOT explicitly.\\n"\n            f"Candidates found:\\n{shown}"\n        )\n    return candidates[0]\n\n\ndef has_dataset_layout(root: Path) -> bool:\n    return all((root / split / "images").is_dir() and (root / split / "labels").is_dir()\n               for split in ("train", "valid", "test"))\n\n\ndef validate_dataset_layout(root: Path) -> None:\n    if not has_dataset_layout(root):\n        raise ValueError(\n            f"{root} must contain train/images, train/labels, valid/images, "\n            "valid/labels, test/images, and test/labels"\n        )\n\n\ndef load_class_names(root: Path) -> list[str]:\n    """Read arbitrary class names from data.yaml without a four-class assumption."""\n    yaml_path = root / "data.yaml"\n    if not yaml_path.is_file():\n        return EXPECTED_CLASSES.copy()\n    try:\n        import yaml\n        document = yaml.safe_load(yaml_path.read_text(encoding="utf-8")) or {}\n        names = document.get("names", [])\n        if isinstance(names, dict):\n            names = [names[key] for key in sorted(names, key=lambda value: int(value))]\n        if isinstance(names, list) and names:\n            result = [str(name) for name in names]\n            declared = document.get("nc")\n            if declared is not None and int(declared) != len(result):\n                raise ValueError(f"data.yaml declares nc={declared} but has {len(result)} names")\n            return result\n    except ImportError:\n        pass\n    # Minimal fallback for environments without PyYAML.\n    found: dict[int, str] = {}\n    in_names = False\n    for line in yaml_path.read_text(encoding="utf-8").splitlines():\n        stripped = line.strip()\n        if stripped.startswith("names:"):\n            in_names = True\n            inline = stripped.split(":", 1)[1].strip()\n            if inline.startswith("[") and inline.endswith("]"):\n                return [part.strip().strip("\'\\" ") for part in inline[1:-1].split(",")]\n            continue\n        if in_names:\n            match = re.match(r"^(\\d+)\\s*:\\s*[\'\\"]?(.+?)[\'\\"]?$", stripped)\n            if match:\n                found[int(match.group(1))] = match.group(2).strip()\n            elif stripped and not line.startswith((" ", "\\t")):\n                break\n    result = [found[i] for i in sorted(found)]\n    return result or EXPECTED_CLASSES.copy()\n\n\ndef polygon_area(points: np.ndarray) -> float:\n    x, y = points[:, 0], points[:, 1]\n    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) / 2.0)\n\n\ndef _orientation(a: np.ndarray, b: np.ndarray, c: np.ndarray, eps: float = 1e-12) -> int:\n    ab, ac = b - a, c - a\n    value = float(ab[0] * ac[1] - ab[1] * ac[0])\n    if abs(value) <= eps:\n        return 0\n    return 1 if value > 0 else -1\n\n\ndef _on_segment(a: np.ndarray, b: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> bool:\n    return (min(a[0], b[0]) - eps <= p[0] <= max(a[0], b[0]) + eps and\n            min(a[1], b[1]) - eps <= p[1] <= max(a[1], b[1]) + eps)\n\n\ndef segments_intersect(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> bool:\n    o1, o2 = _orientation(a, b, c), _orientation(a, b, d)\n    o3, o4 = _orientation(c, d, a), _orientation(c, d, b)\n    if o1 * o2 < 0 and o3 * o4 < 0:\n        return True\n    return ((o1 == 0 and _on_segment(a, b, c)) or\n            (o2 == 0 and _on_segment(a, b, d)) or\n            (o3 == 0 and _on_segment(c, d, a)) or\n            (o4 == 0 and _on_segment(c, d, b)))\n\n\ndef polygon_self_intersects(points: np.ndarray) -> bool:\n    n = len(points)\n    for i in range(n):\n        a, b = points[i], points[(i + 1) % n]\n        for j in range(i + 1, n):\n            if j == i or (j + 1) % n == i or (i + 1) % n == j:\n                continue\n            c, d = points[j], points[(j + 1) % n]\n            if segments_intersect(a, b, c, d):\n                return True\n    return False\n\n\ndef parse_yolo_label(path: Path, n_classes: int, allow_empty: bool = False, allow_self_intersection: bool = False) -> list[tuple[int, np.ndarray]]:\n    if not path.is_file():\n        raise FileNotFoundError(f"Missing label: {path}")\n    polygons: list[tuple[int, np.ndarray]] = []\n    for line_number, raw in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):\n        if not raw.strip():\n            continue\n        tokens = raw.split()\n        if len(tokens) < 7 or (len(tokens) - 1) % 2:\n            raise ValueError(f"{path}:{line_number}: malformed polygon row")\n        try:\n            class_id = int(tokens[0])\n            points = np.asarray([float(v) for v in tokens[1:]], dtype=np.float64).reshape(-1, 2)\n        except Exception as error:\n            raise ValueError(f"{path}:{line_number}: invalid numeric value") from error\n        if class_id not in range(n_classes):\n            raise ValueError(f"{path}:{line_number}: class {class_id} outside 0..{n_classes - 1}")\n        if not np.isfinite(points).all() or (points < 0).any() or (points > 1).any():\n            raise ValueError(f"{path}:{line_number}: coordinates must be finite and in [0,1]")\n        if len(points) >= 4 and np.allclose(points[0], points[-1], rtol=0.0, atol=1e-12):\n            points = points[:-1]\n        if len(np.unique(points, axis=0)) < 3 or polygon_area(points) <= 1e-8:\n            raise ValueError(f"{path}:{line_number}: degenerate polygon")\n        if polygon_self_intersects(points) and not allow_self_intersection:\n            raise ValueError(f"{path}:{line_number}: self-intersecting polygon")\n        polygons.append((class_id, points))\n    if not polygons and not allow_empty:\n        raise ValueError(f"No valid annotations in {path}")\n    return polygons\n\n\ndef normalized_label_text(polygons: list[tuple[int, np.ndarray]]) -> str:\n    rows = []\n    for class_id, points in polygons:\n        coords = " ".join(f"{value:.8f}" for value in points.ravel())\n        rows.append(f"{class_id} {coords}")\n    return "\\n".join(sorted(rows)) + ("\\n" if rows else "")\n\n\ndef rasterize_class_planes(polygons: list[tuple[int, np.ndarray]], size: tuple[int, int],\n                           n_classes: int) -> np.ndarray:\n    width, height = size\n    planes = np.zeros((n_classes, height, width), dtype=np.uint8)\n    for class_id, points in polygons:\n        pixel_points = [(float(x) * width, float(y) * height) for x, y in points]\n        image = Image.fromarray(planes[class_id] * 255, mode="L")\n        ImageDraw.Draw(image).polygon(pixel_points, fill=255, outline=255)\n        planes[class_id] = (np.asarray(image) > 0).astype(np.uint8)\n    return planes\n\n\ndef class_aware_iou(a: list[tuple[int, np.ndarray]], b: list[tuple[int, np.ndarray]],\n                    n_classes: int, raster_size: int = 256) -> float:\n    ma = rasterize_class_planes(a, (raster_size, raster_size), n_classes).astype(bool)\n    mb = rasterize_class_planes(b, (raster_size, raster_size), n_classes).astype(bool)\n    union = np.logical_or(ma, mb).sum()\n    return float(np.logical_and(ma, mb).sum() / union) if union else 1.0\n\n\nclass DisjointSet:\n    def __init__(self, n: int):\n        self.parent = list(range(n))\n        self.rank = [0] * n\n\n    def find(self, value: int) -> int:\n        while self.parent[value] != value:\n            self.parent[value] = self.parent[self.parent[value]]\n            value = self.parent[value]\n        return value\n\n    def union(self, a: int, b: int) -> None:\n        a, b = self.find(a), self.find(b)\n        if a == b:\n            return\n        if self.rank[a] < self.rank[b]:\n            a, b = b, a\n        self.parent[b] = a\n        if self.rank[a] == self.rank[b]:\n            self.rank[a] += 1\n\n\n@dataclass\nclass SourceRecord:\n    old_split: str\n    image_path: Path\n    label_path: Path\n    relative_path: str\n    stem: str\n    source_key: str\n    augmented: bool\n    width: int\n    height: int\n    file_hash: str\n    pixel_hash: str\n    polygons: list[tuple[int, np.ndarray]]\n    classes: tuple[int, ...]\n    source_group_id: str = ""\n    split_group_id: str = ""\n    patient_id: str = ""\n    case_id: str = ""\n    source_collection: str = ""\n    original_url: str = ""\n    image_id: str = ""\n    conflict_min_iou: float = 1.0\n    severe_conflict: bool = False\n    source_class_conflict: bool = False\n    provenance_status: str = ""\n    license_or_permission: str = ""\n\n\ndef scan_dataset(root: Path, names: list[str]) -> list[SourceRecord]:\n    records: list[SourceRecord] = []\n    errors: list[str] = []\n    for split in ("train", "valid", "test"):\n        image_dir, label_dir = root / split / "images", root / split / "labels"\n        images = sorted(p for p in image_dir.iterdir()\n                        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)\n        image_stems = {p.stem for p in images}\n        orphan_labels = sorted(p for p in label_dir.glob("*.txt") if p.stem not in image_stems)\n        errors.extend(f"Orphan label: {p}" for p in orphan_labels)\n        for image_path in images:\n            label_path = label_dir / f"{image_path.stem}.txt"\n            try:\n                # Empty YOLO files are valid background images.  They are kept\n                # for detection training and filtered by MMDetection\'s\n                # filter_empty_gt setting when appropriate.\n                polygons = parse_yolo_label(\n                    label_path, len(names), allow_empty=True, allow_self_intersection=True\n                )\n                with Image.open(image_path) as image:\n                    image.load()\n                    rgb = image.convert("RGB")\n                    width, height = rgb.size\n                    pixel_hash = hashlib.sha256(\n                        f"{width}x{height}|".encode() + rgb.tobytes()\n                    ).hexdigest()\n                records.append(SourceRecord(\n                    old_split=split,\n                    image_path=image_path,\n                    label_path=label_path,\n                    relative_path=str(image_path.relative_to(root)),\n                    stem=image_path.stem,\n                    source_key=inferred_source_key(image_path.stem),\n                    augmented=is_augmented(image_path.stem),\n                    width=width,\n                    height=height,\n                    file_hash=sha256_file(image_path),\n                    pixel_hash=pixel_hash,\n                    polygons=polygons,\n                    classes=tuple(sorted({class_id for class_id, _ in polygons})),\n                ))\n            except Exception as error:\n                errors.append(str(error))\n        print(f"Scanned {split}: {len(images)} images", flush=True)\n    if errors:\n        preview = "\\n".join(errors[:30])\n        raise RuntimeError(f"Input audit failed with {len(errors)} issue(s):\\n{preview}")\n    if not records:\n        raise RuntimeError("No valid input images were found")\n    return records\n\n\ndef load_manual_metadata(path: Path | None) -> tuple[dict[str, dict[str, str]], dict[str, dict[str, str]]]:\n    by_path: dict[str, dict[str, str]] = {}\n    by_source: dict[str, dict[str, str]] = {}\n    for row in read_csv(path):\n        cleaned = {key: (value or "").strip() for key, value in row.items()}\n        rel = cleaned.get("relative_path", "")\n        source = cleaned.get("inferred_source_key", "")\n        if not rel and not source:\n            raise ValueError("Each manual metadata row needs relative_path or inferred_source_key")\n        if rel:\n            by_path[rel] = cleaned\n        if source:\n            by_source[source] = cleaned\n    return by_path, by_source\n\n\ndef assign_source_groups(records: list[SourceRecord], manual_metadata: Path | None) -> None:\n    by_path, by_source = load_manual_metadata(manual_metadata)\n    dsu = DisjointSet(len(records))\n    indexes: dict[tuple[str, str], int] = {}\n    manual_for_record: list[dict[str, str]] = []\n    for index, record in enumerate(records):\n        manual = by_path.get(record.relative_path, by_source.get(record.source_key, {}))\n        manual_for_record.append(manual)\n        keys = [("inferred_source", record.source_key), ("pixel_hash", record.pixel_hash)]\n        manual_source = manual.get("source_photo_id", "")\n        if manual_source:\n            keys.append(("manual_source", manual_source))\n        for key in keys:\n            if key in indexes:\n                dsu.union(index, indexes[key])\n            else:\n                indexes[key] = index\n\n    components: dict[int, list[int]] = defaultdict(list)\n    for index in range(len(records)):\n        components[dsu.find(index)].append(index)\n\n    for members in components.values():\n        evidence = sorted({records[i].source_key for i in members})\n        source_group_id = stable_id("src_", "|".join(evidence))\n        patient_values = {manual_for_record[i].get("patient_id", "") for i in members}\n        case_values = {manual_for_record[i].get("case_id", "") for i in members}\n        patient_values.discard("")\n        case_values.discard("")\n        if len(patient_values) > 1 or len(case_values) > 1:\n            raise ValueError(f"Conflicting patient/case metadata in {source_group_id}")\n        patient_id = next(iter(patient_values), "")\n        case_id = next(iter(case_values), "")\n        for i in members:\n            manual = manual_for_record[i]\n            records[i].source_group_id = source_group_id\n            records[i].patient_id = patient_id\n            records[i].case_id = case_id\n            records[i].source_collection = manual.get("source_collection", "")\n            records[i].original_url = manual.get("original_url", "")\n            records[i].provenance_status = manual.get("provenance_status", "")\n            records[i].license_or_permission = manual.get("license_or_permission", "")\n\n    # Patient-level grouping has priority, followed by case, then source photo.\n    # Values are hashed before export so raw identifiers need not enter manifests.\n    for record in records:\n        if record.patient_id:\n            record.split_group_id = stable_id("patient_", record.patient_id)\n        elif record.case_id:\n            record.split_group_id = stable_id("case_", record.case_id)\n        else:\n            record.split_group_id = record.source_group_id\n\n    grouped: dict[str, list[SourceRecord]] = defaultdict(list)\n    for record in records:\n        grouped[record.source_group_id].append(record)\n    for members in grouped.values():\n        # Different class sets among transformed children are never silently\n        # adjudicated. Multi-class images are allowed when all copies agree.\n        conflict = len({member.classes for member in members}) > 1\n        for member in members:\n            member.source_class_conflict = conflict\n\n\ndef detect_duplicate_annotation_conflicts(records: list[SourceRecord], threshold: float,\n                                          n_classes: int) -> list[dict[str, Any]]:\n    by_pixel_hash: dict[str, list[SourceRecord]] = defaultdict(list)\n    for record in records:\n        by_pixel_hash[record.pixel_hash].append(record)\n    rows: list[dict[str, Any]] = []\n    group_minimum: dict[str, float] = defaultdict(lambda: 1.0)\n    for pixel_hash, members in sorted(by_pixel_hash.items()):\n        if len(members) < 2:\n            continue\n        for left, right in itertools.combinations(members, 2):\n            iou = class_aware_iou(left.polygons, right.polygons, n_classes)\n            group_minimum[left.source_group_id] = min(group_minimum[left.source_group_id], iou)\n            rows.append({\n                "pixel_sha256": pixel_hash,\n                "source_group_id": left.source_group_id,\n                "left_path": left.relative_path,\n                "right_path": right.relative_path,\n                "left_old_split": left.old_split,\n                "right_old_split": right.old_split,\n                "class_aware_mask_iou": f"{iou:.6f}",\n                "severe_conflict": iou < threshold,\n            })\n    for record in records:\n        record.conflict_min_iou = group_minimum[record.source_group_id]\n        record.severe_conflict = record.conflict_min_iou < threshold\n    return rows\n\n\ndef choose_canonical_records(records: list[SourceRecord]) -> list[SourceRecord]:\n    by_source: dict[str, list[SourceRecord]] = defaultdict(list)\n    for record in records:\n        by_source[record.source_group_id].append(record)\n    selected: list[SourceRecord] = []\n    for source_group, members in sorted(by_source.items()):\n        # Prefer a likely original, then the highest pixel count, then a stable path.\n        canonical = sorted(\n            members,\n            key=lambda r: (r.augmented, -(r.width * r.height), r.relative_path.lower()),\n        )[0]\n        canonical.image_id = stable_id("img_", source_group)\n        selected.append(canonical)\n    return selected\n\n\ndef draw_overlay(record: SourceRecord, polygons: list[tuple[int, np.ndarray]], destination: Path) -> None:\n    with Image.open(record.image_path) as source:\n        image = source.convert("RGB")\n    draw = ImageDraw.Draw(image, "RGBA")\n    for class_id, points in polygons:\n        pixels = [(float(x) * image.width, float(y) * image.height) for x, y in points]\n        color = COLORS[class_id]\n        draw.polygon(pixels, fill=(*color, 45), outline=(*color, 255), width=2)\n    display = ImageOps.contain(image, (900, 700))\n    canvas = Image.new("RGB", (display.width, display.height + 38), "white")\n    canvas.paste(display, (0, 38))\n    ImageDraw.Draw(canvas).text(\n        (8, 8),\n        f"{record.image_id} | source={record.source_group_id} | classes={record.classes}",\n        fill="black",\n    )\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(destination, quality=90)\n\n\ndef write_review_bundle(out: Path, canonical: list[SourceRecord], names: list[str],\n                        conflict_rows: list[dict[str, Any]]) -> None:\n    bundle = out / "review_bundle"\n    image_dir, label_dir, overlay_dir = bundle / "images", bundle / "labels", bundle / "overlays"\n    for directory in (image_dir, label_dir, overlay_dir):\n        directory.mkdir(parents=True, exist_ok=True)\n\n    review_rows: list[dict[str, Any]] = []\n    for record in canonical:\n        extension = record.image_path.suffix.lower()\n        shutil.copy2(record.image_path, image_dir / f"{record.image_id}{extension}")\n        (label_dir / f"{record.image_id}.txt").write_text(\n            normalized_label_text(record.polygons), encoding="utf-8"\n        )\n        draw_overlay(record, record.polygons, overlay_dir / f"{record.image_id}.jpg")\n        review_rows.append({\n            "image_id": record.image_id,\n            "source_group_id": record.source_group_id,\n            "split_group_id": record.split_group_id,\n            "original_relative_path": record.relative_path,\n            "classes": ";".join(names[c] for c in record.classes),\n            "likely_original": not record.augmented,\n            "severe_duplicate_label_conflict": record.severe_conflict,\n            "source_family_class_disagreement": record.source_class_conflict,\n            "duplicate_conflict_min_iou": f"{record.conflict_min_iou:.6f}",\n            "status": "pending",\n            "reviewer": "",\n            "target_definition": "",\n            "notes": "",\n        })\n    write_csv(bundle / "review_status.csv", review_rows)\n    write_csv(out / "metadata" / "duplicate_annotation_conflicts.csv", conflict_rows)\n    protocol = """# Annotation protocol — complete before detailed segmentation\n\nDataset version:\nClinical/reviewer lead:\nAnnotation software and version:\nReview and adjudication procedure:\nUncertainty/ignore-region policy:\n\n## Nail psoriasis\nDefine whether the target is the whole affected nail plate or only abnormal tissue.\nState how nail boundaries, surrounding skin, occlusion, and multiple nails are handled.\n\n## SJS-TEN\nDefine the visible manifestation being outlined.  State how oral, ocular, skin,\ndiffuse, uncertain, and non-visible regions are handled.\n\n## Acne\nDefine whether each lesion is an instance or whether the task is an affected-skin\nregion.  Define lesion types and whether intervening healthy skin is excluded.\n\n## Vitiligo\nDefine the depigmented-patch boundary, uncertain transition zones, hair/mucosal\nregions, occlusion, holes, and disconnected components.\n\n## Completion rules\nAll visible targets labeled: yes/no\nEmpty verified-negative images allowed: yes/no\nIndependent evaluation-mask review required: yes\nStatus values in review_status.csv: approved, exclude, pending\n"""\n    (bundle / "annotation_protocol.md").write_text(protocol, encoding="utf-8")\n    (bundle / "README.md").write_text(\n        "Review/correct labels in labels/<image_id>.txt, complete annotation_protocol.md, "\n        "and set status=approved with reviewer and target_definition.  Upload this whole "\n        "folder and rerun the notebook with MODE=\'reviewed\'.  Automated masks are proposals "\n        "until a qualified reviewer approves them.\\n",\n        encoding="utf-8",\n    )\n\n\ndef load_reviewed_labels(reviewed_dir: Path, canonical: list[SourceRecord], n_classes: int,\n                         require_all: bool) -> tuple[list[SourceRecord], dict[str, list[tuple[int, np.ndarray]]]]:\n    status_path = reviewed_dir / "review_status.csv"\n    label_dir = reviewed_dir / "labels"\n    protocol_path = reviewed_dir / "annotation_protocol.md"\n    if not status_path.is_file() or not label_dir.is_dir() or not protocol_path.is_file():\n        raise FileNotFoundError(\n            "Reviewed directory needs review_status.csv, annotation_protocol.md, and labels/"\n        )\n    protocol = protocol_path.read_text(encoding="utf-8")\n    required_fields = [\n        "Dataset version:", "Clinical/reviewer lead:",\n        "Annotation software and version:", "Review and adjudication procedure:",\n        "Uncertainty/ignore-region policy:",\n    ]\n    incomplete = []\n    for field in required_fields:\n        match = re.search(rf"^{re.escape(field)}\\s*(.*)$", protocol, flags=re.MULTILINE)\n        if not match or not match.group(1).strip():\n            incomplete.append(field)\n    if incomplete:\n        raise ValueError(\n            "Complete these annotation_protocol.md fields before reviewed export: "\n            + ", ".join(incomplete)\n        )\n    statuses = {row.get("image_id", "").strip(): row for row in read_csv(status_path)}\n    approved: list[SourceRecord] = []\n    labels: dict[str, list[tuple[int, np.ndarray]]] = {}\n    pending: list[str] = []\n    for record in canonical:\n        row = statuses.get(record.image_id, {})\n        status = (row.get("status", "") or "").strip().lower()\n        if status == "approved":\n            if not (row.get("reviewer", "") or "").strip():\n                raise ValueError(f"Approved {record.image_id} has no reviewer")\n            if not (row.get("target_definition", "") or "").strip():\n                raise ValueError(f"Approved {record.image_id} has no target_definition")\n            label_path = label_dir / f"{record.image_id}.txt"\n            labels[record.image_id] = parse_yolo_label(label_path, n_classes, allow_empty=True)\n            approved.append(record)\n        elif status not in {"exclude", "excluded"}:\n            pending.append(record.image_id)\n    if require_all and pending:\n        raise RuntimeError(f"{len(pending)} canonical cases remain pending review")\n    if not approved:\n        raise RuntimeError("No approved reviewed cases were found")\n    return approved, labels\n\n\ndef build_group_units(records: list[SourceRecord], n_classes: int) -> list[dict[str, Any]]:\n    groups: dict[str, list[SourceRecord]] = defaultdict(list)\n    for record in records:\n        groups[record.split_group_id].append(record)\n    units = []\n    for group_id, members in sorted(groups.items()):\n        class_counts = np.zeros(n_classes, dtype=float)\n        for member in members:\n            for class_id in member.classes:\n                class_counts[class_id] += 1\n        units.append({"group_id": group_id, "records": members,\n                      "n_images": len(members), "class_counts": class_counts})\n    return units\n\n\ndef allocation_cost(image_counts: np.ndarray, class_counts: np.ndarray,\n                    image_targets: np.ndarray, class_targets: np.ndarray) -> float:\n    image_scale = np.maximum(image_targets, 1.0)\n    class_scale = np.maximum(class_targets, 1.0)\n    image_error = np.square((image_counts - image_targets) / image_scale).mean()\n    class_error = np.square((class_counts - class_targets) / class_scale).mean()\n    overshoot = np.square(np.maximum(image_counts - image_targets * 1.15, 0) / image_scale).sum()\n    return float(class_error * 4.0 + image_error + overshoot * 3.0)\n\n\ndef allocate_groups(units: list[dict[str, Any]], fractions: list[float], seed: int,\n                    attempts: int = 200) -> list[int]:\n    if not units or not math.isclose(sum(fractions), 1.0, abs_tol=1e-8):\n        raise ValueError("Allocation needs units and fractions summing to one")\n    n_splits = len(fractions)\n    # The original export has a few hundred groups; this 19-class YOLO export\n    # has thousands of one-image groups.  Full 200-attempt search becomes a\n    # multi-minute quadratic startup with negligible practical improvement.\n    attempts = min(attempts, max(3, 15000 // max(len(units), 1)))\n    total_images = sum(unit["n_images"] for unit in units)\n    total_classes = np.sum([unit["class_counts"] for unit in units], axis=0)\n    image_targets = np.asarray(fractions) * total_images\n    class_targets = np.asarray(fractions)[:, None] * total_classes[None, :]\n    rarity = 1.0 / np.maximum(total_classes, 1.0)\n    best: tuple[float, list[int]] | None = None\n\n    for attempt in range(attempts):\n        rng = random.Random(seed + attempt * 7919)\n        order = list(range(len(units)))\n        rng.shuffle(order)\n        order.sort(\n            key=lambda i: (float(np.dot(units[i]["class_counts"], rarity)),\n                           units[i]["n_images"], rng.random()),\n            reverse=True,\n        )\n        image_counts = np.zeros(n_splits, dtype=float)\n        class_counts = np.zeros((n_splits, len(total_classes)), dtype=float)\n        assignment = [-1] * len(units)\n        for index in order:\n            candidate_splits = list(range(n_splits))\n            rng.shuffle(candidate_splits)\n            scored = []\n            for split in candidate_splits:\n                next_images = image_counts.copy()\n                next_classes = class_counts.copy()\n                next_images[split] += units[index]["n_images"]\n                next_classes[split] += units[index]["class_counts"]\n                scored.append((allocation_cost(next_images, next_classes,\n                                               image_targets, class_targets), split))\n            _, chosen = min(scored)\n            assignment[index] = chosen\n            image_counts[chosen] += units[index]["n_images"]\n            class_counts[chosen] += units[index]["class_counts"]\n\n        missing = int((class_counts == 0).sum())\n        empty = int((image_counts == 0).sum())\n        score = allocation_cost(image_counts, class_counts, image_targets, class_targets)\n        score += missing * 1_000_000 + empty * 1_000_000\n        if best is None or score < best[0]:\n            best = (score, assignment)\n    assert best is not None\n    return best[1]\n\n\ndef create_splits(records: list[SourceRecord], n_classes: int, seed: int,\n                  folds: int) -> tuple[dict[str, str], dict[str, int]]:\n    units = build_group_units(records, n_classes)\n    group_class_presence = np.sum(\n        [unit["class_counts"] > 0 for unit in units], axis=0\n    )\n    if (group_class_presence < max(3, folds)).any():\n        details = {str(i): int(value) for i, value in enumerate(group_class_presence)}\n        raise RuntimeError(\n            f"Too few independent split groups for {folds}-fold class coverage: {details}"\n        )\n\n    dev_test = allocate_groups(units, [0.80, 0.20], seed)\n    main_assignment: dict[str, str] = {}\n    development_units = []\n    for unit, value in zip(units, dev_test):\n        if value == 1:\n            main_assignment[unit["group_id"]] = "test"\n        else:\n            development_units.append(unit)\n    train_valid = allocate_groups(development_units, [0.75, 0.25], seed + 1)\n    for unit, value in zip(development_units, train_valid):\n        main_assignment[unit["group_id"]] = "train" if value == 0 else "valid"\n\n    fold_assignment_values = allocate_groups(development_units, [1.0 / folds] * folds, seed + 2)\n    fold_assignment = {\n        unit["group_id"]: fold for unit, fold in zip(development_units, fold_assignment_values)\n    }\n    return main_assignment, fold_assignment\n\n\ndef update_record_classes(records: list[SourceRecord], labels: dict[str, list[tuple[int, np.ndarray]]]) -> None:\n    for record in records:\n        polygons = labels[record.image_id]\n        record.classes = tuple(sorted({class_id for class_id, _ in polygons}))\n\n\ndef mask_and_overlap(polygons: list[tuple[int, np.ndarray]], size: tuple[int, int],\n                     n_classes: int) -> tuple[np.ndarray, int]:\n    planes = rasterize_class_planes(polygons, size, n_classes).astype(bool)\n    occupancy = planes.sum(axis=0)\n    semantic = np.zeros((size[1], size[0]), dtype=np.uint8)\n    for class_id in range(n_classes):\n        semantic[planes[class_id]] = class_id + 1\n    overlap = occupancy > 1\n    semantic[overlap] = IGNORE_INDEX\n    return semantic, int(overlap.sum())\n\n\ndef export_coco(records: list[SourceRecord], labels: dict[str, list[tuple[int, np.ndarray]]],\n                split: str, destination: Path, names: list[str], file_locations: dict[str, str]) -> dict[str, int]:\n    images, annotations = [], []\n    annotation_id = 1\n    for image_number, record in enumerate(sorted(records, key=lambda r: r.image_id), 1):\n        images.append({\n            "id": image_number,\n            "file_name": file_locations[record.image_id],\n            "width": record.width,\n            "height": record.height,\n            "source_group_id": record.source_group_id,\n            "split_group_id": record.split_group_id,\n        })\n        for class_id, points in labels[record.image_id]:\n            xs = points[:, 0] * record.width\n            ys = points[:, 1] * record.height\n            pixel_points = np.column_stack([xs, ys])\n            single_mask = rasterize_class_planes(\n                [(class_id, points)], (record.width, record.height), len(names)\n            )[class_id]\n            area = int(single_mask.sum())\n            if area <= 0:\n                raise RuntimeError(f"Rasterized zero-area annotation for {record.image_id}")\n            annotations.append({\n                "id": annotation_id,\n                "image_id": image_number,\n                "category_id": class_id + 1,\n                "segmentation": [[float(v) for v in pixel_points.ravel()]],\n                "area": area,\n                "bbox": [float(xs.min()), float(ys.min()),\n                         float(xs.max() - xs.min()), float(ys.max() - ys.min())],\n                "iscrowd": 0,\n            })\n            annotation_id += 1\n    coco = {\n        "info": {"description": f"Skin disease {split}",\n                 "created_utc": datetime.now(timezone.utc).isoformat()},\n        "images": images,\n        "annotations": annotations,\n        "categories": [{"id": i + 1, "name": name, "supercategory": "skin_disease"}\n                       for i, name in enumerate(names)],\n    }\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(coco, indent=2), encoding="utf-8")\n    return {"images": len(images), "annotations": len(annotations)}\n\n\ndef write_yaml(path: Path, dataset_root: Path, names: list[str], train: str,\n               valid: str, test: str | None = None) -> None:\n    lines = [f"path: {dataset_root}", f"train: {train}", f"val: {valid}"]\n    if test:\n        lines.append(f"test: {test}")\n    lines.extend([f"nc: {len(names)}", "names:"])\n    lines.extend(f"  {index}: {name}" for index, name in enumerate(names))\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef export_dataset(out: Path, records: list[SourceRecord],\n                   labels: dict[str, list[tuple[int, np.ndarray]]], names: list[str],\n                   main_assignment: dict[str, str], fold_assignment: dict[str, int],\n                   folds: int, mode: str) -> dict[str, Any]:\n    for split in ("train", "valid", "test"):\n        (out / "images" / split).mkdir(parents=True, exist_ok=True)\n        (out / "labels" / split).mkdir(parents=True, exist_ok=True)\n        (out / "semantic_masks" / split).mkdir(parents=True, exist_ok=True)\n    (out / "coco" / "annotations").mkdir(parents=True, exist_ok=True)\n    (out / "cv").mkdir(parents=True, exist_ok=True)\n\n    by_split: dict[str, list[SourceRecord]] = defaultdict(list)\n    file_locations: dict[str, str] = {}\n    manifest_rows: list[dict[str, Any]] = []\n    overlap_pixels = 0\n    for record in sorted(records, key=lambda r: r.image_id):\n        split = main_assignment[record.split_group_id]\n        extension = record.image_path.suffix.lower()\n        filename = f"{record.image_id}{extension}"\n        image_destination = out / "images" / split / filename\n        label_destination = out / "labels" / split / f"{record.image_id}.txt"\n        mask_destination = out / "semantic_masks" / split / f"{record.image_id}.png"\n        shutil.copy2(record.image_path, image_destination)\n        label_destination.write_text(normalized_label_text(labels[record.image_id]), encoding="utf-8")\n        semantic, overlap = mask_and_overlap(\n            labels[record.image_id], (record.width, record.height), len(names)\n        )\n        Image.fromarray(semantic, mode="L").save(mask_destination)\n        overlap_pixels += overlap\n        by_split[split].append(record)\n        file_locations[record.image_id] = f"images/{split}/{filename}"\n        manifest_rows.append({\n            "image_id": record.image_id,\n            "split": split,\n            "cv_fold": "" if split == "test" else fold_assignment[record.split_group_id],\n            "source_group_id": record.source_group_id,\n            "split_group_id": record.split_group_id,\n            "patient_group_verified": bool(record.patient_id),\n            "old_split": record.old_split,\n            "original_relative_path": record.relative_path,\n            "image_file": file_locations[record.image_id],\n            "label_file": f"labels/{split}/{record.image_id}.txt",\n            "semantic_mask_file": f"semantic_masks/{split}/{record.image_id}.png",\n            "width": record.width,\n            "height": record.height,\n            "classes": ";".join(names[c] for c in record.classes),\n            "file_sha256": record.file_hash,\n            "pixel_sha256": record.pixel_hash,\n            "source_collection": record.source_collection,\n            "original_url": record.original_url,\n            "provenance_status": record.provenance_status,\n            "license_or_permission": record.license_or_permission,\n            "target_mode": mode,\n        })\n    write_csv(out / "metadata" / "manifest.csv", manifest_rows)\n\n    coco_counts = {}\n    for split in ("train", "valid", "test"):\n        coco_counts[split] = export_coco(\n            by_split[split], labels, split,\n            out / "coco" / "annotations" / f"instances_{split}.json",\n            names, file_locations,\n        )\n\n    write_yaml(out / "data.yaml", out.resolve(), names,\n               "images/train", "images/valid", "images/test")\n    (out / "data_portable.yaml").write_text(\n        (out / "data.yaml").read_text(encoding="utf-8").replace(\n            f"path: {out.resolve()}", "path: .", 1\n        ), encoding="utf-8"\n    )\n\n    development = by_split["train"] + by_split["valid"]\n    cv_counts = []\n    for fold in range(folds):\n        fold_dir = out / "cv" / f"fold_{fold}"\n        fold_dir.mkdir(parents=True, exist_ok=True)\n        train_records = [r for r in development if fold_assignment[r.split_group_id] != fold]\n        valid_records = [r for r in development if fold_assignment[r.split_group_id] == fold]\n        train_paths = [str((out / file_locations[r.image_id]).resolve()) for r in train_records]\n        valid_paths = [str((out / file_locations[r.image_id]).resolve()) for r in valid_records]\n        (fold_dir / "train.txt").write_text("\\n".join(train_paths) + "\\n", encoding="utf-8")\n        (fold_dir / "valid.txt").write_text("\\n".join(valid_paths) + "\\n", encoding="utf-8")\n        write_yaml(fold_dir / "data.yaml", out.resolve(), names,\n                   f"cv/fold_{fold}/train.txt", f"cv/fold_{fold}/valid.txt")\n        cv_counts.append({"fold": fold, "train_images": len(train_records),\n                          "valid_images": len(valid_records)})\n\n    return {\n        "split_images": {split: len(by_split[split]) for split in ("train", "valid", "test")},\n        "coco": coco_counts,\n        "cv": cv_counts,\n        "semantic_cross_class_overlap_pixels_set_to_ignore": overlap_pixels,\n    }\n\n\ndef verify_exports(out: Path, records: list[SourceRecord],\n                   main_assignment: dict[str, str], fold_assignment: dict[str, int],\n                   folds: int, names: list[str]) -> dict[str, Any]:\n    by_split: dict[str, list[SourceRecord]] = defaultdict(list)\n    for record in records:\n        by_split[main_assignment[record.split_group_id]].append(record)\n\n    group_sets = {split: {r.split_group_id for r in members}\n                  for split, members in by_split.items()}\n    hash_sets = {split: {r.pixel_hash for r in members}\n                 for split, members in by_split.items()}\n    pairwise_group_overlap = {}\n    pairwise_hash_overlap = {}\n    for a, b in itertools.combinations(("train", "valid", "test"), 2):\n        pairwise_group_overlap[f"{a}_{b}"] = len(group_sets[a] & group_sets[b])\n        pairwise_hash_overlap[f"{a}_{b}"] = len(hash_sets[a] & hash_sets[b])\n    if any(pairwise_group_overlap.values()) or any(pairwise_hash_overlap.values()):\n        raise RuntimeError("Export split leakage detected")\n\n    class_images = {}\n    for split in ("train", "valid", "test"):\n        counts = Counter(class_id for r in by_split[split] for class_id in set(r.classes))\n        class_images[split] = {names[i]: counts[i] for i in range(len(names))}\n        missing = [name for name, value in class_images[split].items() if value == 0]\n        if missing:\n            raise RuntimeError(f"{split} has no images for: {missing}")\n        for record in by_split[split]:\n            image_matches = list((out / "images" / split).glob(f"{record.image_id}.*"))\n            if len(image_matches) != 1:\n                raise RuntimeError(f"Expected one exported image for {record.image_id}")\n            exported_label = out / "labels" / split / f"{record.image_id}.txt"\n            parse_yolo_label(exported_label, len(names), allow_empty=True)\n            mask = np.asarray(Image.open(out / "semantic_masks" / split / f"{record.image_id}.png"))\n            if mask.shape != (record.height, record.width):\n                raise RuntimeError(f"Semantic mask shape mismatch for {record.image_id}")\n\n    cv_report = []\n    test_groups = group_sets["test"]\n    development = by_split["train"] + by_split["valid"]\n    for fold in range(folds):\n        train_groups = {r.split_group_id for r in development\n                        if fold_assignment[r.split_group_id] != fold}\n        valid_groups = {r.split_group_id for r in development\n                        if fold_assignment[r.split_group_id] == fold}\n        overlap = len(train_groups & valid_groups)\n        test_overlap = len((train_groups | valid_groups) & test_groups)\n        if overlap or test_overlap:\n            raise RuntimeError(f"Leakage detected in fold {fold}")\n        cv_report.append({"fold": fold, "train_valid_group_overlap": overlap,\n                          "development_test_group_overlap": test_overlap})\n\n    return {\n        "pairwise_split_group_overlap": pairwise_group_overlap,\n        "pairwise_split_pixel_hash_overlap": pairwise_hash_overlap,\n        "class_images": class_images,\n        "cv": cv_report,\n        "all_checks_passed": True,\n    }\n\n\ndef dataset_card(mode: str, source_root: Path, output_counts: dict[str, Any],\n                 canonical_total: int, excluded_conflicts: int, excluded_class_conflicts: int,\n                 patient_verified: int, total_exported: int) -> str:\n    target = ("approved reviewed polygon masks" if mode == "reviewed" else\n              "existing coarse quadrilateral regions on canonical source photographs")\n    readiness = (\n        "The export uses reviewer-approved replacement labels. Clinical validity still "\n        "depends on the completed protocol, reviewer qualifications, provenance, and the "\n        "availability of patient-level grouping."\n        if mode == "reviewed" else\n        "This export is for coarse localization or weak-supervision development. It must "\n        "not be described as a reviewed detailed lesion-segmentation reference standard."\n    )\n    return f"""# Skin disease dataset v2\n\nCreated UTC: {datetime.now(timezone.utc).isoformat()}\nInput root in this run: `{source_root}`\nTarget mode: `{mode}`\nTarget representation: {target}\n\n{readiness}\n\n## Construction\n\n- One canonical image was selected per inferred/confirmed source-photograph group.\n- Byte/decoded-pixel equality, filename-derived source relationships, and optional\n  manual source metadata were used for grouping.\n- Split grouping uses patient IDs when supplied, then case IDs, then source-photo IDs.\n- Original offline augmented copies are excluded; training augmentation should be online.\n- Development/test and CV assignments are group-disjoint and were verified after export.\n- Cross-class semantic-mask overlap is encoded as ignore value {IGNORE_INDEX}.\n\nCanonical candidates before mode filtering: {canonical_total}\nSevere-conflict source cases excluded in coarse mode: {excluded_conflicts}\nSource-family class-disagreement cases excluded in coarse mode: {excluded_class_conflicts}\nExported cases with verified patient grouping: {patient_verified}/{total_exported}\nOutput counts: `{json.dumps(output_counts[\'split_images\'], sort_keys=True)}`\n\n## Required reporting limitations\n\nFilename-derived source groups are not patient identities. Missing patient IDs, source\nprovenance, permissions, demographic/clinical variables, independent annotations, and\nexternal validation remain limitations unless supplied and documented outside this script.\nThis pipeline cannot infer diagnoses or correct clinical boundaries.\n"""\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default=None,\n                        help="Uploaded Image_Annotation_rebuilt directory; auto-detected if omitted")\n    parser.add_argument("--output-root", default="/kaggle/working/skin_dataset_v2")\n    parser.add_argument("--mode", choices=("coarse", "reviewed"), default="coarse")\n    parser.add_argument("--manual-metadata", default=None,\n                        help="Optional CSV with relative_path/source key and patient/source fields")\n    parser.add_argument("--reviewed-dir", default=None,\n                        help="Completed review_bundle directory; required for reviewed mode")\n    parser.add_argument("--require-all-reviewed", action="store_true")\n    parser.add_argument("--conflict-iou-threshold", type=float, default=0.50)\n    parser.add_argument("--folds", type=int, default=5)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--zip", action="store_true")\n    args = parser.parse_args(argv)\n\n    source_root = find_dataset_root(args.input_root)\n    output_root = Path(args.output_root).expanduser().resolve()\n    if output_root.exists():\n        raise FileExistsError(\n            f"Output already exists: {output_root}. Choose a new OUTPUT_ROOT so an earlier "\n            "dataset version is never silently overwritten."\n        )\n    if not 0 <= args.conflict_iou_threshold <= 1:\n        raise ValueError("Conflict IoU threshold must be between zero and one")\n    if args.folds < 2:\n        raise ValueError("At least two folds are required")\n\n    output_root.mkdir(parents=True)\n    (output_root / "metadata").mkdir()\n    (output_root / "reports").mkdir()\n    names = load_class_names(source_root)\n    records = scan_dataset(source_root, names)\n    assign_source_groups(records, Path(args.manual_metadata).resolve() if args.manual_metadata else None)\n    conflicts = detect_duplicate_annotation_conflicts(\n        records, args.conflict_iou_threshold, len(names)\n    )\n    canonical = choose_canonical_records(records)\n    write_review_bundle(output_root, canonical, names, conflicts)\n\n    source_manifest = []\n    for record in records:\n        source_manifest.append({\n            "relative_path": record.relative_path,\n            "old_split": record.old_split,\n            "inferred_source_key": record.source_key,\n            "source_group_id": record.source_group_id,\n            "split_group_id": record.split_group_id,\n            "patient_group_verified": bool(record.patient_id),\n            "case_group_verified": bool(record.case_id),\n            "augmented_prefix_detected": record.augmented,\n            "canonical": record.image_id != "",\n            "width": record.width,\n            "height": record.height,\n            "classes": ";".join(names[c] for c in record.classes),\n            "file_sha256": record.file_hash,\n            "pixel_sha256": record.pixel_hash,\n            "duplicate_conflict_min_iou": f"{record.conflict_min_iou:.6f}",\n            "severe_duplicate_label_conflict": record.severe_conflict,\n            "source_family_class_disagreement": record.source_class_conflict,\n            "source_collection": record.source_collection,\n            "original_url": record.original_url,\n            "provenance_status": record.provenance_status,\n            "license_or_permission": record.license_or_permission,\n        })\n    write_csv(output_root / "metadata" / "source_manifest.csv", source_manifest)\n    metadata_template = []\n    for record in canonical:\n        metadata_template.append({\n            "relative_path": record.relative_path,\n            "inferred_source_key": record.source_key,\n            "source_photo_id": "",\n            "patient_id": "",\n            "case_id": "",\n            "source_collection": "",\n            "original_url": "",\n            "provenance_status": "unknown",\n            "license_or_permission": "unknown",\n        })\n    write_csv(output_root / "metadata" / "manual_metadata_template.csv", metadata_template)\n\n    if args.mode == "reviewed":\n        if not args.reviewed_dir:\n            raise ValueError("--reviewed-dir is required when --mode reviewed")\n        selected, labels = load_reviewed_labels(\n            Path(args.reviewed_dir).resolve(), canonical, len(names), args.require_all_reviewed\n        )\n        update_record_classes(selected, labels)\n        excluded_conflicts = 0\n        excluded_class_conflicts = 0\n    else:\n        selected = [record for record in canonical\n                    if not record.severe_conflict and not record.source_class_conflict]\n        labels = {record.image_id: record.polygons for record in selected}\n        excluded_conflicts = sum(record.severe_conflict for record in canonical)\n        excluded_class_conflicts = sum(\n            record.source_class_conflict and not record.severe_conflict for record in canonical\n        )\n        if not selected:\n            raise RuntimeError("All canonical sources were excluded by the conflict policy")\n\n    main_assignment, fold_assignment = create_splits(\n        selected, len(names), args.seed, args.folds\n    )\n    output_counts = export_dataset(\n        output_root, selected, labels, names,\n        main_assignment, fold_assignment, args.folds, args.mode\n    )\n    integrity = verify_exports(\n        output_root, selected, main_assignment, fold_assignment, args.folds, names\n    )\n    patient_verified = sum(bool(record.patient_id) for record in selected)\n    report = {\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "input_root": str(source_root),\n        "output_root": str(output_root),\n        "mode": args.mode,\n        "seed": args.seed,\n        "folds": args.folds,\n        "input_images": len(records),\n        "inferred_or_confirmed_source_groups": len({r.source_group_id for r in records}),\n        "canonical_candidates": len(canonical),\n        "exported_canonical_images": len(selected),\n        "severe_conflict_sources_excluded": excluded_conflicts,\n        "source_class_conflict_sources_excluded": excluded_class_conflicts,\n        "patient_grouping_verified_for_all_exported": patient_verified == len(selected),\n        "reviewed_labels_used": args.mode == "reviewed",\n        "detailed_segmentation_claim_ready_from_mode_alone": False,\n        "output_counts": output_counts,\n        "integrity": integrity,\n        "limitations": [\n            "Source-photo inference is not patient identity unless manual metadata supplies it.",\n            "The script cannot verify diagnosis, reviewer expertise, provenance, permissions, or clinical masks.",\n            "Coarse mode retains existing quadrilateral targets and is not detailed lesion segmentation.",\n            "Reviewed mode means the provided review manifest approved the labels; the study must document who reviewed them.",\n        ],\n    }\n    (output_root / "reports" / "rebuild_report.json").write_text(\n        json.dumps(report, indent=2), encoding="utf-8"\n    )\n    (output_root / "dataset_card.md").write_text(\n        dataset_card(args.mode, source_root, output_counts, len(canonical),\n                     excluded_conflicts, excluded_class_conflicts,\n                     patient_verified, len(selected)),\n        encoding="utf-8",\n    )\n    version = {\n        "name": "skin_dataset_v2",\n        "created_utc": report["created_utc"],\n        "mode": args.mode,\n        "seed": args.seed,\n        "folds": args.folds,\n        "manifest_sha256": sha256_file(output_root / "metadata" / "manifest.csv"),\n        "split_definition_sha256": hashlib.sha256(\n            json.dumps({"main": main_assignment, "cv": fold_assignment}, sort_keys=True).encode()\n        ).hexdigest(),\n    }\n    (output_root / "dataset_version.json").write_text(\n        json.dumps(version, indent=2), encoding="utf-8"\n    )\n\n    archive_path = None\n    if args.zip:\n        archive_path = shutil.make_archive(\n            str(output_root), "zip", root_dir=output_root.parent, base_dir=output_root.name\n        )\n    print("\\nREBUILD COMPLETE")\n    print(json.dumps(report, indent=2))\n    if archive_path:\n        print(f"Archive: {archive_path}")\n    if args.mode == "coarse":\n        print("\\nIMPORTANT: This is a group-clean COARSE/WEAK-SUPERVISION export.")\n        print("Complete review_bundle and rerun with MODE=\'reviewed\' for detailed-mask work.")\n    return 0\n\n\n', _KAGGLE_DATASET_REBUILD_V2_module.__file__, 'exec'), _KAGGLE_DATASET_REBUILD_V2_module.__dict__)

SESSION28_BOOTSTRAP_SOURCE = '"""Launch standalone Session 28 in a pinned Kaggle Python 3.11 environment."""\n\nfrom __future__ import annotations\n\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n\nENV_ROOT = Path("/tmp/session28_openmmlab")\nPYTHON = ENV_ROOT / "venv" / "bin" / "python"\nMMCV_WHEEL = (\n    "https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/"\n    "mmcv-2.1.0-cp311-cp311-manylinux1_x86_64.whl"\n)\nPROBE = r\'\'\'import sys\nimport numpy, scipy, cv2, torch, torchvision, mmcv, mmengine, mmdet, yapf\nfrom scipy.sparse import csr_matrix\nfrom mmcv.ops import nms, roi_align\nfrom mmdet.utils import register_all_modules\nassert sys.version_info[:2] == (3, 11), sys.version\nassert numpy.__version__ == \'1.26.4\', numpy.__version__\nassert scipy.__version__ == \'1.14.1\', scipy.__version__\nassert cv2.__version__ == \'4.10.0\', cv2.__version__\nassert torch.__version__.startswith(\'2.1.0\'), torch.__version__\nassert torchvision.__version__.startswith(\'0.16.0\'), torchvision.__version__\nassert torch.version.cuda == \'11.8\', torch.version.cuda\nassert mmcv.__version__ == \'2.1.0\', mmcv.__version__\nassert mmengine.__version__ == \'0.10.7\', mmengine.__version__\nassert mmdet.__version__ == \'3.3.0\', mmdet.__version__\nassert torch.cuda.is_available(), \'CUDA GPU unavailable\'\nregister_all_modules(init_default_scope=True)\nboxes = torch.tensor([[0., 0., 4., 4.]], device=\'cuda\')\nscores = torch.tensor([0.9], device=\'cuda\')\nassert len(nms(boxes, scores, 0.5)[1]) == 1\nfeatures = torch.ones((1, 1, 8, 8), device=\'cuda\', requires_grad=True)\nrois = torch.tensor([[0., 0., 0., 7., 7.]], device=\'cuda\')\nroi_align(features, rois, (4, 4), 1., 0, \'avg\', True).sum().backward()\nassert features.grad is not None\nprint(\'Session 28 pinned stack and CUDA ops passed\')\'\'\'\n\n\ndef environment() -> dict[str, str]:\n    result = os.environ.copy()\n    result["MPLBACKEND"] = "Agg"\n    result["MPLCONFIGDIR"] = str(ENV_ROOT / "matplotlib")\n    result["PYTHONNOUSERSITE"] = "1"\n    result["UV_PYTHON_INSTALL_DIR"] = str(ENV_ROOT / "python")\n    result["UV_CACHE_DIR"] = str(ENV_ROOT / "cache")\n    result["UV_PYTHON_BIN_DIR"] = str(ENV_ROOT / "bin")\n    result.pop("PYTHONPATH", None)\n    result.pop("PYTHONHOME", None)\n    return result\n\n\ndef probe() -> bool:\n    if not PYTHON.is_file():\n        return False\n    try:\n        result = subprocess.run(\n            [str(PYTHON), "-c", PROBE], env=environment(),\n            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n            timeout=180,\n        )\n    except (OSError, subprocess.TimeoutExpired):\n        return False\n    print(result.stdout, flush=True)\n    return result.returncode == 0\n\n\ndef run(command: list[str], timeout: int = 600) -> None:\n    print("  [install]", " ".join(command), flush=True)\n    subprocess.check_call(command, env=environment(), timeout=timeout)\n\n\ndef install() -> None:\n    if not Path("/kaggle/working").is_dir():\n        raise RuntimeError("Session 28 bootstrap is intended for a Kaggle GPU notebook")\n    if shutil.which("nvidia-smi") is None:\n        raise RuntimeError("Enable a Kaggle GPU accelerator before running Session 28")\n    if shutil.disk_usage("/tmp").free < 9 * 1024**3:\n        raise RuntimeError("Need at least 9 GiB free in /tmp for the pinned environment")\n    ENV_ROOT.mkdir(parents=True, exist_ok=True)\n    if PYTHON.parent.parent.exists():\n        shutil.rmtree(PYTHON.parent.parent)\n    installer = ENV_ROOT / "installer"\n    if installer.exists():\n        shutil.rmtree(installer)\n    run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",\n         "--target", str(installer), "uv==0.8.22"], timeout=300)\n    uv = installer / "bin" / "uv"\n    run([str(uv), "venv", "--python", "3.11", "--seed", str(PYTHON.parent.parent)],\n        timeout=420)\n    pip = [str(PYTHON), "-m", "pip", "install", "-q", "--no-cache-dir"]\n    run(pip + ["pip==25.2", "setuptools==75.1.0", "wheel==0.44.0"], timeout=300)\n    run(pip + ["torch==2.1.0", "torchvision==0.16.0", "--index-url",\n               "https://download.pytorch.org/whl/cu118"], timeout=1500)\n    run(pip + [\n        "numpy==1.26.4", "scipy==1.14.1", "opencv-python==4.10.0.84",\n        "matplotlib==3.9.2", "Pillow==10.4.0", "PyYAML==6.0.2",\n        "pycocotools==2.0.10", "shapely==2.0.6", "six==1.16.0",\n        "terminaltables==3.1.10", "tqdm==4.66.5", "addict==2.4.0",\n        "termcolor==2.4.0", "packaging==24.1", "rich==13.9.4",\n        "yapf==0.40.2", "mmengine==0.10.7",\n    ], timeout=600)\n    run(pip + ["--no-deps", MMCV_WHEEL, "mmdet==3.3.0"], timeout=600)\n    run([str(PYTHON), "-m", "pip", "check"], timeout=60)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    args = sys.argv[1:] if argv is None else argv\n    if not args:\n        raise SystemExit("Usage: session28_bootstrap.py RUNTIME_SCRIPT [training args]")\n    runtime = Path(args[0]).resolve()\n    if not runtime.is_file():\n        raise FileNotFoundError(runtime)\n    if not probe():\n        print("  [deps] creating isolated Python 3.11 OpenMMLab stack", flush=True)\n        install()\n        if not probe():\n            raise RuntimeError("Pinned Session 28 dependency/CUDA probe failed")\n    return subprocess.call(\n        [str(PYTHON), "-u", str(runtime), *args[1:]],\n        env={**environment(), "PYTHONUNBUFFERED": "1"},\n    )\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

# END EMBEDDED SESSION 28 HELPERS

from SESSION_24_condinst_r101_dcn_cv import (
    CONDINST_R50_URL,
    EXPECTED_CLASSES,
    _balanced_subset,
    _coco_evaluate,
    _detect_gpu_ids,
    _merge_oof_predictions,
    _parse_folds,
    _run_prediction,
    _run_training,
    _write_prediction_config,
    _yolo_to_coco as _session24_yolo_to_coco,
)

try:
    from KAGGLE_DATASET_REBUILD_V2 import (
        allocate_groups,
        assign_source_groups,
        build_group_units,
        choose_canonical_records,
        create_splits,
        detect_duplicate_annotation_conflicts,
        has_dataset_layout,
        load_class_names,
        scan_dataset,
    )
except ImportError:  # Local workspace import; notebook embeds the top-level name.
    from kaggle_notebooks.KAGGLE_DATASET_REBUILD_V2 import (
        allocate_groups,
        assign_source_groups,
        build_group_units,
        choose_canonical_records,
        create_splits,
        detect_duplicate_annotation_conflicts,
        has_dataset_layout,
        load_class_names,
        scan_dataset,
    )


def _balanced_subset(items: list[dict], limit: int, seed: int) -> list[dict]:
    """Choose a quick smoke subset while covering every class present in labels."""
    if len(items) <= limit:
        return list(items)
    import random
    shuffled = list(items)
    random.Random(seed).shuffle(shuffled)
    class_ids = set()
    parsed = []
    for item in shuffled:
        ids = set()
        for line in item["label"].read_text(encoding="utf-8").splitlines():
            values = line.split()
            if values:
                try:
                    ids.add(int(values[0]))
                except ValueError:
                    pass
        parsed.append((item, ids))
        class_ids.update(ids)
    chosen, used = [], set()
    for class_id in sorted(class_ids):
        for item, ids in parsed:
            if item["key"] not in used and class_id in ids:
                chosen.append(item); used.add(item["key"]); break
        if len(chosen) >= limit:
            return chosen[:limit]
    for item, _ in parsed:
        if len(chosen) >= limit:
            break
        if item["key"] not in used:
            chosen.append(item); used.add(item["key"])
    return chosen


SESSION_NAME = "session28_condinst_r50_fpn_cv_long_v3"
RUNTIME_BUILD_ID = "SESSION28-LONG-V4-ISOLATED-DIRECT"


def _yolo_to_coco(
    items: list[dict], names: list[str], destination: Path, image_ids: dict
) -> dict:
    """Export COCO annotations with the metadata required by pycocotools."""
    report = _session24_yolo_to_coco(items, names, destination, image_ids)
    document = json.loads(destination.read_text(encoding="utf-8"))
    document.setdefault("info", {
        "description": "Session 28 group-disjoint CondInst fold export",
        "version": "1.0",
    })
    document.setdefault("licenses", [])
    destination.write_text(json.dumps(document), encoding="utf-8")
    return report


def _ensure_dependencies() -> None:
    """Require the notebook bootstrap's pinned environment before training."""
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Session 28 needs the pinned Python 3.11 environment. Run the "
            "regenerated SESSION_28_KAGGLE_RUN_THIS.ipynb, not a pasted source cell."
        )
    import numpy
    import scipy
    import torch
    import mmcv
    import mmengine
    import mmdet
    import yapf  # noqa: F401 - MMEngine imports this during model setup.

    expected = {
        "numpy": (numpy.__version__, "1.26.4"),
        "scipy": (scipy.__version__, "1.14.1"),
        "mmcv": (mmcv.__version__, "2.1.0"),
        "mmengine": (mmengine.__version__, "0.10.7"),
        "mmdet": (mmdet.__version__, "3.3.0"),
    }
    wrong = {name: versions for name, versions in expected.items()
             if versions[0] != versions[1]}
    if wrong or not torch.__version__.startswith("2.1.0") or not torch.cuda.is_available():
        raise RuntimeError(f"Session 28 pinned environment check failed: {wrong}")
    print("  [deps] pinned Python 3.11 OpenMMLab environment ready", flush=True)


def _parse_seeds(value: str) -> list[int]:
    seeds = []
    for part in value.split(","):
        part = part.strip()
        if not part:
            continue
        seed = int(part)
        if seed < 0:
            raise ValueError("Seeds must be non-negative integers")
        if seed not in seeds:
            seeds.append(seed)
    if not seeds:
        raise ValueError("--seeds must contain at least one integer")
    return seeds


def _image_count(root: Path, split: str) -> int:
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(
        path.is_file() and path.suffix.lower() in suffixes
        for path in (root / split / "images").iterdir()
    )


def _dataset_counts(root: Path) -> tuple[int, int, int]:
    return tuple(_image_count(root, split) for split in ("train", "valid", "test"))


def _resolve_dataset_root(hint: str) -> Path:
    """Resolve any complete YOLO export, including skin_disease_yolo (19 classes)."""
    def valid(root: Path) -> bool:
        return has_dataset_layout(root) and (root / "data.yaml").is_file()

    def score(root: Path) -> tuple[int, int, int, int, int, str]:
        name = root.name.lower()
        normalized_name = "".join(character for character in name if character.isalnum())
        counts = _dataset_counts(root)
        return (
            int(normalized_name == "skindisnet2annotation"),
            int(name in {"skin_disease_yolo", "skin-disease-yolo"}),
            int(name == "image_annotation_rebuilt"),
            int((root / "metadata" / "manifest.csv").is_file()),
            counts[0],
            str(root),
        )

    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if valid(explicit):
            print(f"  [data] explicit dataset={explicit}; counts={_dataset_counts(explicit)}")
            return explicit
        if explicit.is_dir():
            nested = [path.parent.resolve() for path in explicit.rglob("data.yaml")
                      if valid(path.parent)]
            if len(nested) == 1:
                print(f"  [data] selected nested dataset: {nested[0]}")
                return nested[0]
            preferred = [path for path in nested
                         if path.name.lower() in {"skin_disease_yolo", "skin-disease-yolo",
                                                  "image_annotation_rebuilt"}]
            if len(preferred) == 1:
                print(f"  [data] selected nested dataset: {preferred[0]}")
                return preferred[0]
        raise FileNotFoundError(
            f"DATA_ROOT does not resolve to train/valid/test images+labels: {explicit}"
        )

    search = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd()
    candidates = sorted({path.parent.resolve() for path in search.rglob("data.yaml")
                         if valid(path.parent)}, key=score, reverse=True)
    if not candidates:
        raise FileNotFoundError(
            "No complete YOLO dataset found. Attach Image_Annotation_rebuilt "
            "(or skin_disease_yolo / SkinDisNet_2_Annotation) via Add Data, "
            "or set DATA_ROOT explicitly."
        )
    best_score = score(candidates[0])[:-1]
    tied = [path for path in candidates if score(path)[:-1] == best_score]
    if len(tied) > 1:
        choices = "\n".join(f"    - {path} counts={_dataset_counts(path)}" for path in tied)
        raise RuntimeError("Multiple equally suitable datasets; set DATA_ROOT explicitly:\n" + choices)
    print(f"  [data] selected dataset: {candidates[0]}; counts={_dataset_counts(candidates[0])}")
    return candidates[0]


def _record_item(record) -> dict:
    return {
        "image": record.image_path.resolve(),
        "label": record.label_path.resolve(),
        "key": str(record.image_path.resolve()),
        "source_group_id": record.source_group_id,
        "pixel_hash": record.pixel_hash,
    }


def _apply_manifest_lineage(root: Path, records: list) -> dict[str, int]:
    """Use the dataset manifest to keep each generated variant with its source."""
    manifest_path = root / "manifest.csv"
    if not manifest_path.is_file():
        return {"manifest_records": 0, "augmented_records": 0}

    required_columns = {"name", "split", "source", "augmented"}
    rows: dict[str, dict[str, str]] = {}
    with manifest_path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not required_columns.issubset(reader.fieldnames or ()):
            raise ValueError(
                f"{manifest_path} must contain columns: "
                f"{', '.join(sorted(required_columns))}"
            )
        for row in reader:
            name = (row.get("name") or "").strip()
            if not name or name in rows:
                raise ValueError(f"Missing or duplicate image name in {manifest_path}: {name!r}")
            rows[name] = row

    matched: set[str] = set()
    augmented_records = 0
    for record in records:
        row = rows.get(record.stem)
        if row is None:
            raise ValueError(
                f"Missing manifest lineage for {record.relative_path} in {manifest_path}"
            )
        source = (row.get("source") or "").strip()
        split = (row.get("split") or "").strip().lower()
        augmented = (row.get("augmented") or "").strip().lower()
        if not source or split != record.old_split or augmented not in {"true", "false"}:
            raise ValueError(
                f"Inconsistent manifest lineage for {record.relative_path}: {row}"
            )
        record.source_key = source
        record.augmented = augmented == "true"
        matched.add(record.stem)
        augmented_records += record.augmented

    unmatched_rows = set(rows) - matched
    if unmatched_rows:
        raise ValueError(
            f"{len(unmatched_rows)} manifest entries do not match dataset images; "
            f"examples: {sorted(unmatched_rows)[:5]}"
        )
    return {
        "manifest_records": len(rows),
        "augmented_records": augmented_records,
    }


def _prepare_direct_group_folds(
    root: Path, names: list[str], seed: int, folds: int = 5,
    locked_test: bool = True,
) -> tuple[list[dict], list[dict], dict]:
    """Build folds in memory from the direct dataset without another export."""
    records = scan_dataset(root, names)
    lineage = _apply_manifest_lineage(root, records)
    assign_source_groups(records, manual_metadata=None)
    conflicts = detect_duplicate_annotation_conflicts(
        records, threshold=0.50, n_classes=len(names)
    )
    canonical_all = choose_canonical_records(records)
    retained_canonical = [
        record for record in canonical_all
        if not record.severe_conflict and not record.source_class_conflict
    ]
    if not retained_canonical:
        raise RuntimeError("All inferred source groups were excluded by conflict checks")

    retained_groups = {record.source_group_id for record in retained_canonical}
    retained_records = [
        record for record in records if record.source_group_id in retained_groups
    ]
    if locked_test:
        main_assignment, fold_assignment = create_splits(
            retained_canonical, len(names), seed, folds
        )
        development_groups = {
            group for group, split in main_assignment.items() if split != "test"
        }
        test_groups = {
            group for group, split in main_assignment.items() if split == "test"
        }
    else:
        # Pool every retained source group into the folds. Use this for
        # datasets whose received splits leak across boundaries (e.g.
        # Image_Annotation_rebuilt), matching the S25/S34/S35 protocol.
        units = build_group_units(retained_canonical, len(names))
        fold_values = allocate_groups(units, [1.0 / folds] * folds, seed + 2)
        fold_assignment = {
            unit["group_id"]: fold for unit, fold in zip(units, fold_values)
        }
        development_groups = set(fold_assignment)
        test_groups = set()
    canonical_by_group = {
        record.source_group_id: record for record in retained_canonical
    }
    fold_data = []
    for fold in range(folds):
        valid_groups = {
            group for group in development_groups if fold_assignment[group] == fold
        }
        train_groups = development_groups - valid_groups
        train = [
            _record_item(record) for record in retained_records
            if record.source_group_id in train_groups
        ]
        # Evaluation uses one canonical image per independent source group.
        valid = [
            _record_item(canonical_by_group[group]) for group in sorted(valid_groups)
        ]
        valid_class_images = {
            names[class_id]: sum(
                class_id in canonical_by_group[group].classes
                for group in valid_groups
            )
            for class_id in range(len(names))
        }
        if any(value == 0 for value in valid_class_images.values()):
            message = (
                f"Fold {fold} is missing a validation class: {valid_class_images}"
            )
            if lineage["manifest_records"]:
                raise RuntimeError(message)
            print(f"  [data] warning: {message} "
                  "(filename-derived groups; continuing)", flush=True)
        train_hashes = {item["pixel_hash"] for item in train}
        valid_hashes = {item["pixel_hash"] for item in valid}
        if train_groups & valid_groups or train_hashes & valid_hashes:
            raise RuntimeError(f"Leakage detected while constructing fold {fold}")
        fold_data.append({
            "fold": fold,
            "train": train,
            "valid": valid,
            "train_source_groups": len(train_groups),
            "valid_source_groups": len(valid_groups),
            "valid_class_images": valid_class_images,
        })

    test = [
        _record_item(canonical_by_group[group]) for group in sorted(test_groups)
    ]
    test_class_images = {
        names[class_id]: sum(
            class_id in canonical_by_group[group].classes for group in test_groups
        )
        for class_id in range(len(names))
    }
    if test_groups and any(value == 0 for value in test_class_images.values()):
        raise RuntimeError(f"Locked test is missing a class: {test_class_images}")
    development_hashes = {
        record.pixel_hash for record in retained_records
        if record.source_group_id in development_groups
    }
    test_hashes = {item["pixel_hash"] for item in test}
    if development_groups & test_groups or development_hashes & test_hashes:
        raise RuntimeError("Development/test source or pixel leakage detected")

    report = {
        "mode": (
            "direct internal group reconstruction"
            if locked_test else
            "direct internal group reconstruction (all groups pooled into CV)"
        ),
        "input_split_images": {
            split: _image_count(root, split) for split in ("train", "valid", "test")
        },
        "input_images": len(records),
        **lineage,
        "inferred_source_groups": len({r.source_group_id for r in records}),
        "canonical_candidates": len(canonical_all),
        "retained_source_groups": len(retained_canonical),
        "retained_training_images_including_augmentations": len(retained_records),
        "severe_conflict_groups_excluded": sum(r.severe_conflict for r in canonical_all),
        "class_conflict_groups_excluded": sum(
            r.source_class_conflict and not r.severe_conflict for r in canonical_all
        ),
        "duplicate_conflict_pairs_reviewed": len(conflicts),
        "development_source_groups": len(development_groups),
        "locked_test_source_groups": len(test_groups),
        "locked_test_class_images": test_class_images,
        "validation_policy": "one canonical image per source group",
        "training_policy": "all retained images/augmentations from training groups",
        "group_disjoint_verified": True,
        "pixel_hash_disjoint_verified": True,
        "patient_identity_verified": False,
        "folds": [
            {
                "fold": item["fold"],
                "train_images": len(item["train"]),
                "train_source_groups": item["train_source_groups"],
                "valid_images": len(item["valid"]),
                "valid_source_groups": item["valid_source_groups"],
                "valid_class_images": item["valid_class_images"],
            }
            for item in fold_data
        ],
    }
    return fold_data, test, report


def _write_training_config(
    destination: Path,
    train_json: Path,
    valid_json: Path,
    work_dir: Path,
    checkpoint: Path,
    names: list[str],
    gpu_count: int,
    max_iters: int,
    image_size: int,
    batch_per_gpu: int,
    accumulation: int,
    seed: int,
    *,
    mask_out_stride: int = 4,
    mask_loss_weight: float = 2.0,
    max_masks_to_train: int = 150,
    score_thr: float = 0.02,
    mask_thr: float = 0.5,
    scale_range: tuple[float, float] = (0.6, 1.5),
    photometric: bool = False,
    base_lr: float | None = None,
    scheduler: str = "cosine",
    warmup_iters: int = 1000,
) -> None:
    """Write a complete MMDetection 3.3 CondInst R50-FPN configuration."""
    classes = repr(tuple(names))
    palette = repr([(220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230)])
    checkpoint_interval = min(max_iters, 2000)
    warmup = min(warmup_iters, max(10, max_iters // 10))
    effective_batch = gpu_count * batch_per_gpu * accumulation
    learning_rate = base_lr if base_lr else 0.01 * effective_batch / 16.0
    milestone_1 = max(warmup + 1, int(max_iters * 0.80))
    milestone_2 = max(milestone_1 + 1, int(max_iters * 0.92))
    if scheduler == "cosine":
        decay_schedule = (
            f"    dict(type='CosineAnnealingLR', by_epoch=False,\n"
            f"         begin={warmup}, end={max_iters}, T_max={max_iters - warmup},\n"
            f"         eta_min={learning_rate * 0.01!r}),"
        )
    else:
        decay_schedule = (
            f"    dict(type='MultiStepLR', begin=0, end={max_iters}, by_epoch=False,\n"
            f"         milestones=[{milestone_1}, {milestone_2}], gamma=0.1),"
        )
    colour_jitter = (
        "    dict(type='PhotoMetricDistortion', brightness_delta=24,\n"
        "         contrast_range=(0.85, 1.15), saturation_range=(0.85, 1.15),\n"
        "         hue_delta=9),\n"
        if photometric else ""
    )

    config = f'''# Auto-generated by SESSION 28.
_base_ = ['mmdet::condinst/condinst_r50_fpn_ms-poly-90k_coco_instance.py']

default_scope = 'mmdet'
metainfo = dict(classes={classes}, palette={palette})
image_size = ({image_size}, {image_size})

# Genuine CondInst R50-FPN.  No DCN, attention block, or fallback detector.
model = dict(
    backbone=dict(
        depth=50,
        frozen_stages=1,
        with_cp=False,
        init_cfg=dict(type='Pretrained', checkpoint='torchvision://resnet50'),
    ),
    bbox_head=dict(num_classes={len(names)}),
    mask_head=dict(
        max_masks_to_train={max_masks_to_train},
        mask_out_stride={mask_out_stride},
        loss_mask=dict(type='DiceLoss', use_sigmoid=True, activate=True,
                       eps=5e-6, loss_weight={mask_loss_weight}),
    ),
    test_cfg=dict(score_thr={score_thr}, mask_thr={mask_thr}),
)

train_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=True),
    dict(type='RandomFlip', prob=0.5),
{colour_jitter}    dict(type='RandomResize', scale=image_size,
         ratio_range=({scale_range[0]}, {scale_range[1]}),
         resize_type='Resize', keep_ratio=True),
    dict(type='RandomCrop', crop_size=image_size, crop_type='absolute',
         recompute_bbox=True, allow_negative_crop=False),
    dict(type='FilterAnnotations', min_gt_bbox_wh=(1.0, 1.0), by_mask=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='Resize', scale=image_size, keep_ratio=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs', meta_keys=(
        'img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

train_dataloader = dict(
    _delete_=True, batch_size={batch_per_gpu}, num_workers=2,
    persistent_workers=True, sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset', data_root='', ann_file={str(train_json)!r},
        data_prefix=dict(img=''), metainfo=metainfo,
        filter_cfg=dict(filter_empty_gt=True, min_size=1),
        pipeline=train_pipeline,
    ),
)
# The held-out fold is evaluated once with the fixed final checkpoint. It is
# never used to select a checkpoint or tune the LR schedule during training.
val_dataloader = None
val_evaluator = None
test_dataloader = dict(
    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,
    drop_last=False, sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', data_root='', ann_file={str(valid_json)!r},
        data_prefix=dict(img=''), metainfo=metainfo, test_mode=True,
        pipeline=test_pipeline,
    ),
)
test_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(valid_json)!r},
    metric=['bbox', 'segm'], classwise=True,
)

optim_wrapper = dict(
    _delete_=True, type='AmpOptimWrapper', loss_scale='dynamic',
    accumulative_counts={accumulation},
    optimizer=dict(type='SGD', lr={learning_rate}, momentum=0.9,
                   weight_decay=0.0001),
    clip_grad=dict(max_norm=35.0, norm_type=2),
)
param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False,
         begin=0, end={warmup}),
{decay_schedule}
]
train_cfg = dict(
    _delete_=True, type='IterBasedTrainLoop', max_iters={max_iters},
    val_interval={max_iters + 1},
)
val_cfg = None
test_cfg = dict(_delete_=True, type='TestLoop')
default_hooks = dict(
    logger=dict(type='LoggerHook', interval=50),
    checkpoint=dict(
        type='CheckpointHook', by_epoch=False, interval={checkpoint_interval},
        save_last=True,
        max_keep_ckpts=2,
    ),
)
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
randomness = dict(seed={seed}, deterministic=False, diff_rank_seed=True)
env_cfg = dict(
    cudnn_benchmark=True,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)
model_wrapper_cfg = dict(
    type='MMDistributedDataParallel',
    find_unused_parameters=True,
)
launcher = 'none'
work_dir = {str(work_dir)!r}
load_from = {str(checkpoint)!r}
resume = False
'''
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(config, encoding="utf-8")


def _validate_model_build(config: Path, number_of_classes: int) -> None:
    """Fail before training unless the built model is genuine CondInst R50-FPN."""
    code = r'''
import sys
from mmengine.config import Config
from mmdet.registry import MODELS
from mmdet.utils import register_all_modules
from mmcv.ops import ModulatedDeformConv2dPack
register_all_modules(init_default_scope=True)
cfg = Config.fromfile(sys.argv[1])
expected_classes = int(sys.argv[2])
assert cfg.model_wrapper_cfg.type == 'MMDistributedDataParallel'
assert cfg.model_wrapper_cfg.find_unused_parameters is True
model = MODELS.build(cfg.model)
assert model.__class__.__name__ == 'CondInst'
assert model.backbone.__class__.__name__ == 'ResNet'
assert model.backbone.depth == 50
assert tuple(model.backbone.stage_blocks) == (3, 4, 6, 3)
assert model.neck.__class__.__name__ == 'FPN'
assert model.bbox_head.__class__.__name__ == 'CondInstBboxHead'
assert model.mask_head.__class__.__name__ == 'CondInstMaskHead'
assert model.bbox_head.num_classes == expected_classes
assert model.bbox_head.num_params == 169
assert model.mask_head.num_params == 169
dcn = sum(isinstance(module, ModulatedDeformConv2dPack)
          for module in model.modules())
assert dcn == 0, f'plain R50 baseline unexpectedly contains {dcn} DCNv2 layers'
print('architecture=CondInst; backbone=ResNet-50; neck=FPN; '
      'DCNv2=0; dynamic_mask_parameters=169')
'''
    import subprocess

    subprocess.check_call(
        [sys.executable, "-c", code, str(config), str(number_of_classes)]
    )


def _prepare_condinst_checkpoint(
    work: Path, config: Path
) -> tuple[Path, dict]:
    """Retain exact-shape tensors from the official COCO CondInst-R50 model."""
    prepared = work / "pretrained" / "condinst_r50_coco_for_dataset_classes.pth"
    report_path = work / "pretrained" / "checkpoint_adaptation.json"
    if (
        prepared.is_file()
        and prepared.stat().st_size > 1_000_000
        and report_path.is_file()
    ):
        return prepared, json.loads(report_path.read_text(encoding="utf-8"))

    raw = work / "pretrained" / "condinst_r50_coco_raw.pth"
    raw.parent.mkdir(parents=True, exist_ok=True)
    print("  [weights] downloading official MMDetection CondInst-R50 COCO weights")
    try:
        urlretrieve(CONDINST_R50_URL, raw)
    except Exception as error:
        raw.unlink(missing_ok=True)
        raise RuntimeError(
            "Official CondInst checkpoint download failed. Enable Kaggle Internet; "
            "Session 28 will not substitute another model."
        ) from error
    if not raw.is_file() or raw.stat().st_size <= 1_000_000:
        raw.unlink(missing_ok=True)
        raise RuntimeError("Downloaded CondInst checkpoint is unexpectedly small")

    import torch
    from mmengine.config import Config
    from mmdet.registry import MODELS
    from mmdet.utils import register_all_modules

    register_all_modules(init_default_scope=True)
    model = MODELS.build(Config.fromfile(str(config)).model)
    target = model.state_dict()
    document = torch.load(raw, map_location="cpu", weights_only=False)
    source_state = document.get("state_dict", document.get("model", document))
    source_state = {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in source_state.items()
    }
    retained = {
        key: value
        for key, value in source_state.items()
        if key in target and tuple(value.shape) == tuple(target[key].shape)
    }
    mismatched = sorted(
        key
        for key, value in source_state.items()
        if key in target and tuple(value.shape) != tuple(target[key].shape)
    )
    required = {
        "backbone": any(key.startswith("backbone.") for key in retained),
        "fpn": any(key.startswith("neck.") for key in retained),
        "bbox_head": any(key.startswith("bbox_head.") for key in retained),
        "mask_head": any(key.startswith("mask_head.") for key in retained),
        "dynamic_controller": any("bbox_head.controller" in key for key in retained),
    }
    if not all(required.values()) or len(retained) < 100:
        raise RuntimeError(
            f"Official checkpoint coverage is incomplete: {required}; "
            f"retained={len(retained)}"
        )
    torch.save(
        {
            "state_dict": retained,
            "meta": {
                "source": CONDINST_R50_URL,
                "adaptation": (
                    "Retained every exact-shape CondInst R50-FPN tensor and "
                    "removed COCO-class-dependent tensors with shape mismatch."
                ),
            },
        },
        prepared,
    )
    raw.unlink(missing_ok=True)
    report = {
        "source_url": CONDINST_R50_URL,
        "retained_tensors": len(retained),
        "shape_mismatch_tensors_removed": mismatched,
        "required_components_present": required,
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        f"  [weights] retained={len(retained)} tensors; "
        f"class-dependent mismatches removed={len(mismatched)}"
    )
    return prepared, report


def _marker_path(work_dir: Path) -> Path:
    return work_dir / "SESSION_28_TRAINING_COMPLETE.json"


def _latest_iteration_checkpoint(work_dir: Path) -> Path | None:
    checkpoints = list(work_dir.glob("iter_*.pth"))
    if not checkpoints:
        return None
    return max(checkpoints, key=lambda path: int(path.stem.split("_")[-1]))


def _final_checkpoint(work_dir: Path, max_iters: int) -> Path | None:
    checkpoint = work_dir / f"iter_{max_iters}.pth"
    return checkpoint if checkpoint.is_file() else None


def _completed_checkpoint(work_dir: Path, config: Path) -> Path | None:
    marker = _marker_path(work_dir)
    if not marker.is_file():
        return None
    try:
        record = json.loads(marker.read_text(encoding="utf-8"))
        checkpoint = Path(record["checkpoint"])
        digest = hashlib.sha256(config.read_bytes()).hexdigest()
        if record.get("config_sha256") == digest and checkpoint.is_file():
            return checkpoint
    except (KeyError, OSError, ValueError, json.JSONDecodeError):
        pass
    return None


def _mark_training_complete(
    work_dir: Path, config: Path, checkpoint: Path
) -> None:
    _marker_path(work_dir).write_text(
        json.dumps(
            {
                "checkpoint": str(checkpoint),
                "config_sha256": hashlib.sha256(config.read_bytes()).hexdigest(),
                "completed_utc": datetime.now(timezone.utc).isoformat(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def _mean_std(values: list[float]) -> dict[str, float | None]:
    return {
        "mean": float(statistics.mean(values)),
        "std": float(statistics.stdev(values)) if len(values) > 1 else None,
    }


def _environment_versions() -> dict[str, str | bool | None]:
    import mmcv
    import mmdet
    import mmengine
    import torch
    import torchvision

    return {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "mmcv": mmcv.__version__,
        "mmengine": mmengine.__version__,
        "mmdetection": mmdet.__version__,
    }


def _runtime_source_text() -> str:
    """Recover the complete source when it was pasted into a notebook cell."""
    candidates = []
    if globals().get("__file__"):
        candidates.append(Path(__file__))
    candidates.append(Path("/kaggle/working/SESSION_28_runtime.py"))
    for candidate in candidates:
        try:
            source = candidate.read_text(encoding="utf-8")
        except (OSError, UnicodeError):
            continue
        if RUNTIME_BUILD_ID in source and "def main(" in source:
            return source
    try:
        history = get_ipython().user_ns.get("In", [])  # type: ignore[name-defined]
    except Exception:
        history = []
    for source in reversed(history):
        if (isinstance(source, str) and RUNTIME_BUILD_ID in source
                and "def main(" in source):
            return source
    raise RuntimeError(
        "Could not recover the pasted Session 28 source. Upload the regenerated "
        "SESSION_28_KAGGLE_RUN_THIS.ipynb instead."
    )


def _launch_isolated(args: argparse.Namespace) -> int:
    working = Path("/kaggle/working")
    if not working.is_dir():
        raise RuntimeError("The isolated Session 28 launcher requires Kaggle")
    runtime = working / "SESSION_28_runtime.py"
    bootstrap = working / "SESSION_28_bootstrap.py"
    source = _runtime_source_text()
    compile(source, str(runtime), "exec")
    runtime.write_text(source, encoding="utf-8")
    bootstrap.write_text(SESSION28_BOOTSTRAP_SOURCE, encoding="utf-8")
    command = [
        sys.executable, "-u", str(bootstrap), str(runtime),
        "--data-root", args.data_root,
        "--device", args.device,
        "--folds", args.folds,
        "--seeds", args.seeds,
        "--split-seed", str(args.split_seed),
        "--max-iters", str(args.max_iters),
        "--image-size", str(args.image_size),
        "--batch-per-gpu", str(args.batch_per_gpu),
        "--accumulation", str(args.accumulation),
        "--mask-out-stride", str(args.mask_out_stride),
        "--mask-loss-weight", str(args.mask_loss_weight),
        "--max-masks-to-train", str(args.max_masks_to_train),
        "--score-thr", str(args.score_thr),
        "--mask-thr", str(args.mask_thr),
        "--scale-min", str(args.scale_min),
        "--scale-max", str(args.scale_max),
        "--scheduler", args.scheduler,
        "--warmup-iters", str(args.warmup_iters),
        "--time-budget-hours", str(args.time_budget_hours),
    ]
    if args.base_lr is not None:
        command += ["--base-lr", str(args.base_lr)]
    if args.photometric:
        command.append("--photometric")
    if args.work_dir:
        command += ["--work-dir", args.work_dir]
    if args.locked_test:
        command.append("--locked-test")
    if args.quick:
        command.append("--quick")
    if not args.skip_existing:
        command.append("--no-skip-existing")
    if args.resume:
        command.append("--resume")
    if args.skip_train:
        command.append("--skip-train")
    print("  [deps] launching Session 28 in isolated Python 3.11", flush=True)
    return subprocess.call(command)


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 28: CondInst R50-FPN five-fold multi-seed OOF CV"
    )
    parser.add_argument("--data-root", default="auto")
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--device", default="auto", help="auto, 0, or 0,1")
    parser.add_argument("--folds", default="0,1,2,3,4")
    parser.add_argument("--seeds", default="42")
    parser.add_argument("--split-seed", type=int, default=42)
    parser.add_argument("--max-iters", type=int, default=20000)
    parser.add_argument("--image-size", type=int, default=640)
    parser.add_argument("--batch-per-gpu", type=int, default=2)
    parser.add_argument("--accumulation", type=int, default=1)
    parser.add_argument("--mask-out-stride", type=int, default=4,
                        help="4 is the CondInst default; 2 doubles mask "
                             "resolution and mask memory")
    parser.add_argument("--mask-loss-weight", type=float, default=2.0)
    parser.add_argument("--max-masks-to-train", type=int, default=150)
    parser.add_argument("--score-thr", type=float, default=0.02)
    parser.add_argument("--mask-thr", type=float, default=0.5)
    parser.add_argument("--scale-min", type=float, default=0.6)
    parser.add_argument("--scale-max", type=float, default=1.5)
    parser.add_argument("--photometric", action="store_true")
    parser.add_argument("--base-lr", type=float, default=None,
                        help="Overrides the linear batch-scaled learning rate")
    parser.add_argument("--scheduler", choices=("cosine", "multistep"),
                        default="cosine")
    parser.add_argument("--warmup-iters", type=int, default=1000)
    parser.add_argument("--time-budget-hours", type=float, default=11.0,
                        help="Stop before starting a fold that would not finish")
    parser.add_argument("--locked-test", action="store_true",
                        help="hold out 20%% of source groups as a locked test "
                             "set; default pools all groups into the 5-fold CV")
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--skip-existing", action="store_true", default=True)
    parser.add_argument(
        "--no-skip-existing", dest="skip_existing", action="store_false"
    )
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--skip-train", action="store_true")
    args, _ = parser.parse_known_args(argv)

    if args.max_iters < 1:
        raise ValueError("--max-iters must be positive")
    if args.image_size < 384 or args.image_size % 32:
        raise ValueError("--image-size must be >=384 and divisible by 32")
    if args.batch_per_gpu < 1 or args.accumulation < 1:
        raise ValueError("Batch size and accumulation must be positive")
    if args.mask_out_stride not in (1, 2, 4, 8):
        raise ValueError("--mask-out-stride must divide the mask feature stride 8")
    if not 0.1 <= args.scale_min <= args.scale_max <= 2.5:
        raise ValueError("--scale-min/--scale-max must satisfy 0.1 <= min <= max <= 2.5")
    if args.time_budget_hours <= 0:
        raise ValueError("--time-budget-hours must be positive")

    if Path("/kaggle/working").is_dir():
        if sys.version_info[:2] != (3, 11):
            return _launch_isolated(args)
        try:
            _ensure_dependencies()
        except (RuntimeError, ImportError):
            return _launch_isolated(args)
    elif sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Session 28 needs Python 3.11 with the pinned OpenMMLab stack."
        )

    multiprocessing.freeze_support()
    started = time.time()
    print("=" * 78)
    print("SESSION 28 — CondInst R50-FPN, group-safe five-fold multi-seed CV")
    print("=" * 78)
    _ensure_dependencies()
    gpu_ids = _detect_gpu_ids(args.device)
    selected_folds = _parse_folds(args.folds)
    selected_seeds = _parse_seeds(args.seeds)
    print(
        f"  [run] GPUs={gpu_ids}; folds={selected_folds}; seeds={selected_seeds}"
    )
    config_kwargs = dict(
        mask_out_stride=args.mask_out_stride,
        mask_loss_weight=args.mask_loss_weight,
        max_masks_to_train=args.max_masks_to_train,
        score_thr=args.score_thr,
        mask_thr=args.mask_thr,
        scale_range=(args.scale_min, args.scale_max),
        photometric=args.photometric,
        base_lr=args.base_lr,
        scheduler=args.scheduler,
        warmup_iters=args.warmup_iters,
    )
    print(f"  [run] training knobs: {config_kwargs}")

    root = _resolve_dataset_root(args.data_root)
    names = load_class_names(root)
    fold_data, test_items, integrity = _prepare_direct_group_folds(
        root, names, seed=args.split_seed, folds=5,
        locked_test=args.locked_test,
    )
    print(
        f"  [data] input={integrity['input_images']} images; "
        f"retained_groups={integrity['retained_source_groups']}; "
        f"locked_test_groups={len(test_items)}; direct group folds verified"
    )

    if args.quick:
        selected_folds = [selected_folds[0]]
        selected_seeds = [selected_seeds[0]]
        args.max_iters = min(args.max_iters, 20)
        print("  [quick] one seed, one fold, balanced subset, 20 iterations")

    dataset_slug = "".join(
        character.lower() if character.isalnum() else "_"
        for character in root.name
    ).strip("_")
    default_work = Path("/kaggle/working") / (
        f"{SESSION_NAME}_{dataset_slug}"
        f"{'_quick' if args.quick else ''}"
    )
    work = Path(args.work_dir).expanduser().resolve() if args.work_dir else default_work
    work.mkdir(parents=True, exist_ok=True)
    coco_dir = work / "coco"
    configs_dir = work / "configs"
    predictions_dir = work / "oof_predictions"
    for directory in (coco_dir, configs_dir, predictions_dir):
        directory.mkdir(parents=True, exist_ok=True)

    selected_data = []
    for fold in selected_folds:
        current = dict(fold_data[fold])
        if args.quick:
            current["train"] = _balanced_subset(current["train"], 48, selected_seeds[0])
            current["valid"] = _balanced_subset(current["valid"], 24, selected_seeds[0] + 1)
        selected_data.append(current)

    all_items_by_key = {
        item["key"]: item
        for current in selected_data
        for item in current["train"] + current["valid"]
    }
    image_ids = {
        key: index + 1 for index, key in enumerate(sorted(all_items_by_key))
    }
    oof_items_by_key = {
        item["key"]: item for current in selected_data for item in current["valid"]
    }
    oof_items = [oof_items_by_key[key] for key in sorted(oof_items_by_key)]
    oof_gt = coco_dir / "selected_oof_ground_truth.json"
    oof_conversion = _yolo_to_coco(oof_items, names, oof_gt, image_ids)

    bootstrap_config = configs_dir / "architecture_preflight.py"
    prepared_path = work / "pretrained" / "condinst_r50_coco_for_dataset_classes.pth"
    _write_training_config(
        bootstrap_config,
        oof_gt,
        oof_gt,
        work / "preflight",
        prepared_path,
        names,
        len(gpu_ids),
        args.max_iters,
        args.image_size,
        args.batch_per_gpu,
        args.accumulation,
        selected_seeds[0],
        **config_kwargs,
    )
    compile(bootstrap_config.read_text(encoding="utf-8"), str(bootstrap_config), "exec")
    _validate_model_build(bootstrap_config, len(names))
    pretrained, checkpoint_report = _prepare_condinst_checkpoint(
        work, bootstrap_config
    )

    summary = {
        "session": "SESSION_28",
        "model": "CondInst R50-FPN",
        "framework": "MMDetection 3.3.0",
        "environment": _environment_versions(),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(root),
        "classes": names,
        "selected_folds": selected_folds,
        "seeds": selected_seeds,
        "dataset_integrity": integrity,
        "oof_conversion": oof_conversion,
        "checkpoint_adaptation": checkpoint_report,
        "configuration": {
            "max_iters_per_seed_fold": args.max_iters,
            "image_size": args.image_size,
            "gpus": gpu_ids,
            "batch_per_gpu": args.batch_per_gpu,
            "gradient_accumulation": args.accumulation,
            "optimizer": "SGD with linear warmup and multistep decay",
            "checkpoint_selection": "fixed final iteration; held-out fold never selects weights",
            "pretraining": "official MMDetection CondInst R50 COCO checkpoint",
            "split_seed": args.split_seed,
            "quick": args.quick,
        },
        "seed_results": [],
        "multi_seed_summary": {},
        "locked_test": {
            "images": len(test_items),
            "evaluated": False,
            "reason": (
                "The test split stays untouched during CV and method selection. "
                "Evaluate one frozen final procedure once in a separate final stage."
                if args.locked_test else
                "All source groups are pooled into five-fold CV; the received "
                "test split is not independent for this dataset."
            ),
        },
        "limitations": [
            "Existing coarse labels are quadrilaterals, so metrics measure coarse region "
            "localization rather than clinician-reviewed lesion boundaries.",
            "Source groups are inferred from filenames and exact decoded pixels; they "
            "are not verified patient identities.",
            "All retained augmented images are used only for training. Validation and "
            "locked test use one canonical image per inferred source group.",
            "Three seeds times five folds equals 15 trainings and normally needs "
            "multiple Kaggle sessions.",
            "The pinned Python 3.11 environment uses the official MMCV 2.1 wheel "
            "with MMDetection 3.3 and is separate from Kaggle's notebook Python.",
        ],
    }
    summary_path = work / "SESSION_28_results.json"

    def save_summary() -> None:
        summary_path.write_text(
            json.dumps(summary, indent=2, default=str), encoding="utf-8"
        )

    fold_durations: list[float] = []
    budget_exhausted = False
    for seed in selected_seeds:
        if budget_exhausted:
            break
        print("\n" + "#" * 78)
        print(f"SEED {seed}")
        print("#" * 78)
        prediction_files: list[Path] = []
        fold_results = []

        for current in selected_data:
            fold = current["fold"]
            run_seed = seed + fold
            print("\n" + "-" * 78)
            print(
                f"SEED {seed} / FOLD {fold}: "
                f"train={len(current['train'])}, valid={len(current['valid'])}"
            )
            print("-" * 78)
            fold_coco = coco_dir / f"fold_{fold}"
            fold_config_dir = configs_dir / f"seed_{seed}" / f"fold_{fold}"
            fold_work = work / "runs" / f"seed_{seed}" / f"fold_{fold}"
            for directory in (fold_coco, fold_config_dir, fold_work):
                directory.mkdir(parents=True, exist_ok=True)
            train_json = fold_coco / "train.json"
            valid_json = fold_coco / "valid.json"
            train_report = _yolo_to_coco(
                current["train"], names, train_json, image_ids
            )
            valid_report = _yolo_to_coco(
                current["valid"], names, valid_json, image_ids
            )
            config = fold_config_dir / "condinst_r50_fpn.py"
            _write_training_config(
                config,
                train_json,
                valid_json,
                fold_work,
                pretrained,
                names,
                len(gpu_ids),
                args.max_iters,
                args.image_size,
                args.batch_per_gpu,
                args.accumulation,
                run_seed,
                **config_kwargs,
            )
            compile(config.read_text(encoding="utf-8"), str(config), "exec")

            checkpoint = _latest_iteration_checkpoint(fold_work)
            completed = _completed_checkpoint(fold_work, config)
            if args.skip_train:
                if completed is None:
                    raise FileNotFoundError(
                        f"No config-matched completed checkpoint exists for seed {seed}, fold {fold}"
                    )
                checkpoint = completed
            elif args.skip_existing and completed is not None:
                checkpoint = completed
                print(f"  [train] reusing verified checkpoint: {checkpoint}")
            else:
                if args.resume and _marker_path(fold_work).is_file() and completed is None:
                    raise RuntimeError(
                        f"Cannot resume seed {seed}, fold {fold}: an older completed "
                        "run used a different training configuration. Use the new "
                        "long_v2 work directory."
                    )
                spent = (time.time() - started) / 3600.0
                if fold_durations:
                    projected = spent + max(fold_durations) * 1.05
                    if projected > args.time_budget_hours:
                        print(
                            f"  [budget] stopping before seed {seed} fold {fold}: "
                            f"{spent:.2f}h spent, one more fold would reach "
                            f"~{projected:.2f}h against a {args.time_budget_hours:.2f}h "
                            "budget. Rerun with the same work directory restored "
                            "to continue from here."
                        )
                        budget_exhausted = True
                        break
                fold_started = time.time()
                _run_training(config, fold_work, gpu_ids,
                              args.resume and checkpoint is not None)
                fold_durations.append((time.time() - fold_started) / 3600.0)
                checkpoint = _final_checkpoint(fold_work, args.max_iters)
                if checkpoint is None:
                    raise RuntimeError(
                        f"Seed {seed}, fold {fold} finished without iter_{args.max_iters}.pth"
                    )
                _mark_training_complete(fold_work, config, checkpoint)

            prefix = predictions_dir / f"seed_{seed}_fold_{fold}_valid"
            prediction_config = fold_config_dir / "predict_valid.py"
            _write_prediction_config(config, prediction_config, valid_json, prefix)
            _run_prediction(
                prediction_config, checkpoint, fold_work, gpu_ids[0]
            )
            prediction_json = Path(f"{prefix}.segm.json")
            if not prediction_json.is_file():
                raise FileNotFoundError(
                    f"MMDetection did not create {prediction_json}"
                )
            metrics = _coco_evaluate(valid_json, prediction_json, names)
            prediction_files.append(prediction_json)
            fold_results.append(
                {
                    "fold": fold,
                    "training_seed": run_seed,
                    "checkpoint": str(checkpoint),
                    "train_conversion": train_report,
                    "valid_conversion": valid_report,
                    "metrics": metrics,
                }
            )
            # Preserve completed folds even if the next fold is interrupted.
            summary["seed_results"] = [
                result for result in summary["seed_results"]
                if result["seed"] != seed
            ] + [{"seed": seed, "fold_results": list(fold_results),
                  "complete_five_fold_oof": False}]
            save_summary()

        if not prediction_files:
            print(f"  [budget] seed {seed} produced no completed fold; stopping")
            break
        completed_folds = sorted(item["fold"] for item in fold_results)
        seed_oof_predictions = predictions_dir / f"seed_{seed}_oof.segm.json"
        detection_count = _merge_oof_predictions(
            prediction_files, seed_oof_predictions
        )
        if completed_folds == selected_folds:
            seed_gt = oof_gt
        else:
            partial_by_key = {
                item["key"]: item
                for current in selected_data
                if current["fold"] in completed_folds
                for item in current["valid"]
            }
            partial_items = [partial_by_key[key] for key in sorted(partial_by_key)]
            seed_gt = coco_dir / f"seed_{seed}_partial_oof_ground_truth.json"
            _yolo_to_coco(partial_items, names, seed_gt, image_ids)
            print(
                f"  [oof] seed {seed} is partial; evaluating folds "
                f"{completed_folds} against {len(partial_items)} images"
            )
        seed_metrics = _coco_evaluate(seed_gt, seed_oof_predictions, names)
        fold_scores = [
            item["metrics"]["segm_mAP50"] for item in fold_results
        ]
        summary["seed_results"] = [
            result for result in summary["seed_results"]
            if result["seed"] != seed
        ] + [
            {
                "seed": seed,
                "fold_results": fold_results,
                "oof_prediction_file": str(seed_oof_predictions),
                "oof_detections": detection_count,
                "oof_metrics": seed_metrics,
                "fold_segm_mAP50": _mean_std(fold_scores),
                "completed_folds": completed_folds,
                "complete_five_fold_oof": (
                    completed_folds == [0, 1, 2, 3, 4] and not args.quick
                ),
            }
        ]
        save_summary()

    metric_keys = ["segm_mAP50", "segm_mAP50_95", "bbox_mAP50"]
    scored_seeds = [
        result for result in summary["seed_results"] if "oof_metrics" in result
    ]
    if not scored_seeds:
        save_summary()
        raise RuntimeError(
            "No seed completed a fold within the time budget. Restore this work "
            "directory in a new session and rerun to continue."
        )
    summary["multi_seed_summary"] = {
        key: _mean_std([result["oof_metrics"][key] for result in scored_seeds])
        for key in metric_keys
    }
    summary["multi_seed_summary"]["per_class_segm_mAP50"] = {
        name: _mean_std(
            [
                result["oof_metrics"]["per_class_mAP50"][name]
                for result in scored_seeds
            ]
        )
        for name in names
    }
    summary["multi_seed_summary"]["complete"] = (
        all(result.get("complete_five_fold_oof") for result in scored_seeds)
        and len(scored_seeds) >= 3
        and not args.quick
    )
    summary["multi_seed_summary"]["scored_seeds"] = [
        result["seed"] for result in scored_seeds
    ]
    summary["configuration"].update(
        {
            "mask_out_stride": args.mask_out_stride,
            "mask_loss_weight": args.mask_loss_weight,
            "max_masks_to_train": args.max_masks_to_train,
            "test_score_thr": args.score_thr,
            "test_mask_thr": args.mask_thr,
            "train_scale_jitter": [args.scale_min, args.scale_max],
            "photometric_distortion": args.photometric,
            "lr_schedule": args.scheduler,
            "warmup_iters": args.warmup_iters,
            "base_lr_override": args.base_lr,
            "time_budget_hours": args.time_budget_hours,
            "budget_exhausted": budget_exhausted,
        }
    )
    summary["elapsed_hours"] = (time.time() - started) / 3600.0
    save_summary()

    archive = Path("/kaggle/working/SESSION_28_LONG_V3_results.zip")
    if not archive.parent.is_dir():
        archive = work / "SESSION_28_LONG_V3_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output_zip:
        output_zip.write(summary_path, arcname=summary_path.name)
        output_zip.write(oof_gt, arcname=oof_gt.name)
        adaptation = work / "pretrained" / "checkpoint_adaptation.json"
        output_zip.write(adaptation, arcname=adaptation.name)
        for result in scored_seeds:
            path = Path(result["oof_prediction_file"])
            output_zip.write(path, arcname=f"oof/{path.name}")
        for seed in selected_seeds:
            for fold in selected_folds:
                config = configs_dir / f"seed_{seed}" / f"fold_{fold}" / "condinst_r50_fpn.py"
                if config.is_file():
                    output_zip.write(
                        config,
                        arcname=f"configs/seed_{seed}/fold_{fold}/{config.name}",
                    )

    print("\n" + "=" * 78)
    print("SESSION 28 COMPLETE")
    print("=" * 78)
    for key in metric_keys:
        values = summary["multi_seed_summary"][key]
        if values["std"] is None:
            print(f"{key}: {values['mean']:.4f} (one seed; variability unavailable)")
        else:
            print(f"{key}: {values['mean']:.4f} +/- {values['std']:.4f}")
    print("\nPer-class OOF segm mAP50:")
    for name, values in summary["multi_seed_summary"]["per_class_segm_mAP50"].items():
        if values["std"] is None:
            print(f"  {name}: {values['mean']:.4f}")
        else:
            print(f"  {name}: {values['mean']:.4f} +/- {values['std']:.4f}")
    gap = (
        summary["multi_seed_summary"]["bbox_mAP50"]["mean"]
        - summary["multi_seed_summary"]["segm_mAP50"]["mean"]
    )
    print(
        f"\nbbox-minus-segm mAP50 gap: {gap:.4f} "
        "(a large gap means the mask branch, not the detector, is the bottleneck)"
    )
    if budget_exhausted:
        print(
            "NOTE: the time budget stopped this run early. Restore this work "
            "directory in a new session and rerun the same command to continue."
        )
    print(f"Results JSON: {summary_path}")
    print(f"Results ZIP : {archive}")
    print(f"Checkpoints : {work / 'runs'} (download separately)")
    print("Locked test: untouched")
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    exit_code = main()
    try:
        sys.exit(exit_code)
    except SystemExit as error:
        if error.code:
            raise

  [deps] launching Session 28 in isolated Python 3.11
  [deps] creating isolated Python 3.11 OpenMMLab stack
  [install] /usr/bin/python3 -m pip install -q --no-cache-dir --target /tmp/session28_openmmlab/installer uv==0.8.22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 326.9 MB/s eta 0:00:00
  [install] /tmp/session28_openmmlab/installer/bin/uv venv --python 3.11 --seed /tmp/session28_openmmlab/venv


Using CPython 3.11.13
Creating virtual environment with seed packages at: /tmp/session28_openmmlab/venv
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /tmp/session28_openmmlab/venv/bin/activate


  [install] /tmp/session28_openmmlab/venv/bin/python -m pip install -q --no-cache-dir pip==25.2 setuptools==75.1.0 wheel==0.44.0
  [install] /tmp/session28_openmmlab/venv/bin/python -m pip install -q --no-cache-dir torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
  [install] /tmp/session28_openmmlab/venv/bin/python -m pip install -q --no-cache-dir numpy==1.26.4 scipy==1.14.1 opencv-python==4.10.0.84 matplotlib==3.9.2 Pillow==10.4.0 PyYAML==6.0.2 pycocotools==2.0.10 shapely==2.0.6 six==1.16.0 terminaltables==3.1.10 tqdm==4.66.5 addict==2.4.0 termcolor==2.4.0 packaging==24.1 rich==13.9.4 yapf==0.40.2 mmengine==0.10.7
  [install] /tmp/session28_openmmlab/venv/bin/python -m pip install -q --no-cache-dir --no-deps https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/mmcv-2.1.0-cp311-cp311-manylinux1_x86_64.whl mmdet==3.3.0
  [install] /tmp/session28_openmmlab/venv/bin/python -m pip check
No broken requirements found.
Session 28 pinned stack and CUDA o

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/tmp/session28_openmmlab/venv/lib/python3.11/site-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(
/tmp/session28_openmmlab/venv/lib/python3.11/site-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(


10/17 10:36:52 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.13 (main, Sep 18 2025, 19:46:39) [Clang 20.1.4 ]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: cc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


... (25 outputs left out)

/tmp/session28_openmmlab/venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/tmp/session28_openmmlab/venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[W reducer.cpp:1346] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forwar

10/17 14:56:04 - mmengine - INFO - Iter(train) [   50/20000]  lr: 1.2500e-04  eta: 2:28:28  time: 0.4465  data_time: 0.0137  memory: 9485  grad_norm: inf  loss: 2.4087  loss_cls: 1.1397  loss_bbox: 0.2752  loss_centerness: 0.6283  loss_mask: 0.3655
10/17 14:56:17 - mmengine - INFO - Iter(train) [  100/20000]  lr: 2.5000e-04  eta: 1:57:05  time: 0.2596  data_time: 0.0130  memory: 1398  grad_norm: 7.8336  loss: 2.1609  loss_cls: 0.9367  loss_bbox: 0.2585  loss_centerness: 0.6142  loss_mask: 0.3515
10/17 14:56:30 - mmengine - INFO - Iter(train) [  150/20000]  lr: 3.7500e-04  eta: 1:46:04  time: 0.2557  data_time: 0.0129  memory: 1397  grad_norm: 7.3069  loss: 1.7456  loss_cls: 0.5779  loss_bbox: 0.2546  loss_centerness: 0.6035  loss_mask: 0.3096
10/17 14:56:42 - mmengine - INFO - Iter(train) [  200/20000]  lr: 5.0000e-04  eta: 1:39:00  time: 0.2382  data_time: 0.0132  memory: 1368  grad_norm: 8.0029  loss: 1.5769  loss_cls: 0.4394  loss_bbox: 0.2436  loss_centerness: 0.5822  loss_mask: 0.

/tmp/session28_openmmlab/venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


10/17 15:59:50 - mmengine - INFO - Iter(test) [50/54]    eta: 0:00:00  time: 0.1773  data_time: 0.0036  memory: 8907  
10/17 15:59:51 - mmengine - INFO - results are saved in /kaggle/working/session28_condinst_r50_fpn_cv_long_v3_image_annotation_rebuilt/oof_predictions
10/17 15:59:51 - mmengine - INFO - Iter(test) [54/54]    data_time: 0.0035  time: 0.1700
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.15s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.495
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.747
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.526
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.018
 Average Precision  (AP) @